# Yoga Posture Assessment System: AI-Based Implementation and Evaluation Notebook

## Thesis Project Implementation

This notebook presents the complete implementation pipeline of the proposed AI-powered Yoga Posture Assessment System. The system combines computer vision, biomechanical analysis, and deep learning techniques to automatically evaluate yoga postures from video inputs.

The implementation covers the complete machine learning workflow, starting from human pose extraction using MediaPipe, biomechanical feature generation, temporal sequence preparation, dataset construction, model development, training, evaluation, and deployment.

The developed system aims to analyze yoga movements, classify performed poses, assess posture safety, and provide explainable feedback regarding potential biomechanical risks.

## System Objectives

The proposed system is designed to:

- Extract human body landmarks from yoga videos using MediaPipe Pose estimation.
- Generate biomechanical features representing posture alignment and movement characteristics.
- Learn temporal movement patterns through a deep learning-based sequence model.
- Classify different yoga postures based on extracted motion information.
- Assess whether a performed posture is safe or potentially risky.
- Provide explainable risk analysis based on biomechanical measurements.
- Deploy the trained model through a FastAPI-based inference service.

## Implementation Pipeline

The notebook follows the complete development workflow:

1. Environment Setup and Configuration
2. Human Pose Landmark Extraction
3. Biomechanical Feature Engineering
4. Temporal Sequence Construction
5. Dataset Preparation
6. Feature Processing and Label Encoding
7. Deep Learning Model Development
8. Model Training and Validation
9. Leave-One-Person-Out (LOPO) Evaluation
10. Model Saving and Artifact Management
11. Video-Based Posture Prediction
12. Explainable Risk Assessment
13. Deployment as an API Service

This notebook serves as the core implementation component of the thesis, demonstrating the integration of computer vision, biomechanics, and artificial intelligence for automated yoga posture assessment.

# 1. Environment Setup and Configuration

This section prepares the computational environment required for developing and evaluating the Yoga Posture Assessment System.

The environment configuration includes installing required dependencies, connecting storage resources, importing necessary libraries, and defining the computational settings used throughout the experiment.

The implementation uses GPU acceleration when available to improve training efficiency, especially during deep learning model optimization.

# Yoga Posture Assessment System: AI-Based Implementation and Evaluation Notebook

## Thesis Project Implementation

This notebook presents the complete implementation pipeline of the proposed AI-powered Yoga Posture Assessment System. The system combines computer vision, biomechanical analysis, and deep learning techniques to automatically evaluate yoga postures from video inputs.

The implementation covers the complete machine learning workflow, starting from human pose extraction using MediaPipe, biomechanical feature generation, temporal sequence preparation, dataset construction, model development, training, evaluation, and deployment.

The developed system aims to analyze yoga movements, classify performed poses, assess posture safety, and provide explainable feedback regarding potential biomechanical risks.

## System Objectives

The proposed system is designed to:

- Extract human body landmarks from yoga videos using MediaPipe Pose estimation.
- Generate biomechanical features representing posture alignment and movement characteristics.
- Learn temporal movement patterns through a deep learning-based sequence model.
- Classify different yoga postures based on extracted motion information.
- Assess whether a performed posture is safe or potentially risky.
- Provide explainable risk analysis based on biomechanical measurements.
- Deploy the trained model through a FastAPI-based inference service.

## Implementation Pipeline

The notebook follows the complete development workflow:

1. Environment Setup and Configuration
2. Human Pose Landmark Extraction
3. Biomechanical Feature Engineering
4. Temporal Sequence Construction
5. Dataset Preparation
6. Feature Processing and Label Encoding
7. Deep Learning Model Development
8. Model Training and Validation
9. Leave-One-Person-Out (LOPO) Evaluation
10. Model Saving and Artifact Management
11. Video-Based Posture Prediction
12. Explainable Risk Assessment
13. Deployment as an API Service

This notebook serves as the core implementation component of the thesis, demonstrating the integration of computer vision, biomechanics, and artificial intelligence for automated yoga posture assessment.

# 1. Environment Setup and Configuration

This section prepares the computational environment required for developing and evaluating the Yoga Posture Assessment System.

The environment configuration includes installing required dependencies, connecting storage resources, importing necessary libraries, and defining the computational settings used throughout the experiment.

The implementation uses GPU acceleration when available to improve training efficiency, especially during deep learning model optimization.

In [ ]:
# ============================================================
# Cell 0: Initial Environment Setup
# ============================================================
# This cell prepares the Google Colab environment by installing
# the required dependencies for MediaPipe Pose Landmarker.
#
# Note:
# - This cell only needs to be executed once per new Colab runtime.
# - A runtime restart is required after installation for the
#   newly installed packages to be loaded correctly.
# ============================================================

# Run only once

# Step 1: Install a compatible protobuf version required by MediaPipe.
!pip install -q protobuf==5.29.6

# Step 2: Install MediaPipe and its required dependencies.
# The '--no-deps' flag avoids dependency conflicts, while the
# remaining packages are installed manually.
!pip install -q mediapipe==0.10.21 --no-deps
!pip install -q absl-py>=2.0 flatbuffers>=23.5.26 attrs>=19.1.0

# Step 3: Download the MediaPipe Pose Landmarker model if it
# does not already exist in the current working directory.
import urllib.request, os

if not os.path.exists("pose_landmarker.task"):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        "pose_landmarker.task"
    )
    print("Model downloaded")

# Inform the user that installation has finished.
# Restarting the runtime is necessary before proceeding.
print("Install done — now restart runtime")

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

# Location of the yoga video dataset.
DATASET_PATH    = "/content/drive/MyDrive/Thesis_Dataset"

# Output file for extracted MediaPipe landmarks.
OUTPUT_CSV      = "/content/yoga_video_keypoints.csv"

# Output file for engineered biomechanical features.
BIO_CSV         = "/content/yoga_bio_features.csv"

# Number of frames each video will be normalized to during
# temporal sequence construction.
TARGET_FRAMES   = 60

In [ ]:
# ── Standard library ──
import os
import warnings
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")

# ── Numerical & Data ──
import numpy as np
import pandas as pd

# ── Visualization ──
import matplotlib.pyplot as plt

# ── Computer Vision & Pose ──
import cv2
import mediapipe as mp

# ── ML / Preprocessing ──
import joblib
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix

# ── Deep Learning ──
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Progress Bar ──
from tqdm import tqdm
import time

print("All imports successful!")
print(f"MediaPipe: {mp.__version__}")
print(f"PyTorch:   {torch.__version__}")
print(f"Device:    {'cuda' if torch.cuda.is_available() else 'cpu'}")
#3

In [ ]:
# Select the available computation device.
# GPU (CUDA) is preferred for faster model training; otherwise,
# the CPU will be used.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Training Hyperparameters
# -------------------------------

# Total number of complete passes through the training dataset.
EPOCHS        = 60

# Number of training samples processed in each mini-batch.
BATCH_SIZE    = 16

# Initial learning rate used by the optimizer.
LR            = 1e-4

# Loss weight assigned to the yoga pose classification task.
LAMBDA_POSE   = 0.3

# Loss weight assigned to the posture risk assessment task.
LAMBDA_RISK   = 0.7

# Enable Automatic Mixed Precision (AMP) when GPU acceleration
# is available to improve computational performance.
USE_AMP       = torch.cuda.is_available()

# Initialize the gradient scaler for mixed precision training.
# The scaler is only created when AMP is enabled.
scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

# Display the selected training configuration.
print(f"Device     : {device}")
print(f"Epochs     : {EPOCHS}")
print(f"Batch Size : {BATCH_SIZE}")
print(f"LR         : {LR}")
print(f"AMP        : {USE_AMP}")

# 2. Human Pose Landmark Extraction Using MediaPipe

This section implements the first stage of the proposed system: extracting human body landmarks from yoga video data.

MediaPipe Pose is utilized as the computer vision framework for detecting anatomical keypoints from each video frame. These landmarks represent important body locations, including joints and body segments, which serve as the foundation for further biomechanical analysis.

The extracted landmarks are stored and processed as structured pose data for feature engineering and temporal movement analysis.

An alternative workflow is also provided to load previously extracted landmarks, reducing processing time when repeating experiments.

# 2. Human Pose Landmark Extraction Using MediaPipe

This section implements the first stage of the proposed system: extracting human body landmarks from yoga video data.

MediaPipe Pose is utilized as the computer vision framework for detecting anatomical keypoints from each video frame. These landmarks represent important body locations, including joints and body segments, which serve as the foundation for further biomechanical analysis.

The extracted landmarks are stored and processed as structured pose data for feature engineering and temporal movement analysis.

An alternative workflow is also provided to load previously extracted landmarks, reducing processing time when repeating experiments.

In [ ]:
# Import required libraries for file handling, video processing,
# data storage, and pose estimation.
import os
import cv2
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

# Initialize the MediaPipe Pose solution.
mp_pose = mp.solutions.pose

# Define the directory where extracted landmark data will be saved.
OUTPUT_ROOT = "/content/YogaModel_SavedArtifacts"

# Output CSV file containing all extracted pose landmarks.
LANDMARK_OUTPUT = os.path.join(
    OUTPUT_ROOT,
    "landmarks",
    "yoga_video_keypoints.csv"
)

# Create the output directory if it does not already exist.
os.makedirs(os.path.dirname(LANDMARK_OUTPUT), exist_ok=True)

# Container for storing landmark information extracted from
# every processed frame.
rows = []

# Supported video file formats.
video_extensions = (".mp4", ".avi", ".mov", ".mkv", ".MOV")

# Initialize the MediaPipe Pose detector.
# The selected parameters prioritize landmark accuracy while
# maintaining reliable tracking across video frames.
with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as detector:

    # Iterate through each yoga pose category contained in
    # the dataset directory.
    for pose_label in sorted(os.listdir(DATASET_PATH)):
        pose_path = os.path.join(DATASET_PATH, pose_label)

        # Skip non-directory files.
        if not os.path.isdir(pose_path):
            continue

        # Process every video belonging to the current pose.
        for filename in tqdm(sorted(os.listdir(pose_path)), desc=pose_label):

            # Ignore unsupported file formats.
            if not filename.lower().endswith(video_extensions):
                continue

            video_path = os.path.join(pose_path, filename)
            fn_lower = filename.lower()

            # --------------------------------------------------
            # Determine the posture safety label based on the
            # filename naming convention.
            # --------------------------------------------------
            if "_safe" in fn_lower:
                risk_label = "safe"
            elif any(x in fn_lower for x in ["_unsafe", "_danger", "_warning"]):
                risk_label = "unsafe"
            else:
                risk_label = "unknown"

            # --------------------------------------------------
            # Extract the participant identifier from the filename.
            # This information is later used for
            # Leave-One-Person-Out (LOPO) validation.
            # --------------------------------------------------
            participant_id = None
            for token in filename.replace("-", "_").split("_"):
                if token.lower().startswith("p") and token[1:].isdigit():
                    participant_id = token.upper()
                    break

            # --------------------------------------------------
            # Identify the camera viewpoint based on the filename.
            # Multiple viewing angles improve dataset diversity.
            # --------------------------------------------------
            camera_view = "unknown"

            if "front" in fn_lower:
                camera_view = "front"
            elif "side" in fn_lower:
                camera_view = "side"
            elif "oblique" in fn_lower:
                camera_view = "oblique"
            elif "back" in fn_lower:
                camera_view = "back"

            # Open the current video for frame-by-frame processing.
            cap = cv2.VideoCapture(video_path)

            frame_id = 0

            while cap.isOpened():

                # Read the next video frame.
                ret, frame = cap.read()

                # Stop when no more frames are available.
                if not ret:
                    break

                # Convert OpenCV's BGR image into RGB format,
                # which is required by MediaPipe.
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Perform pose estimation on the current frame.
                result = detector.process(frame_rgb)

                # Continue only if body landmarks were successfully detected.
                if result.pose_landmarks:

                    # Store metadata associated with the current frame.
                    row = {
                        "video_name": filename,
                        "frame_id": frame_id,
                        "pose_label": pose_label,
                        "risk_label": risk_label,
                        "participant_id": participant_id,
                        "camera_view": camera_view
                    }

                    # Save the x, y, z coordinates and visibility score
                    # for each detected landmark.
                    for i, lm in enumerate(result.pose_landmarks.landmark):
                        row[f"x{i}"], row[f"y{i}"], row[f"z{i}"], row[f"v{i}"] = (
                            lm.x,
                            lm.y,
                            lm.z,
                            lm.visibility
                        )

                    # Store the processed frame.
                    rows.append(row)

                # Advance to the next frame.
                frame_id += 1

            # Release the video resource after processing.
            cap.release()

# Convert all extracted landmark records into a DataFrame.
df_kp = pd.DataFrame(rows)

# Save the extracted landmarks for later stages of the pipeline.
df_kp.to_csv(LANDMARK_OUTPUT, index=False)

# Display the output location.
print("\nSaved:", LANDMARK_OUTPUT)

In [ ]:
 (Alternate) – Load Landmarks If Already Extracted

# Run this instead of Step 1 if landmarks
# were already extracted and saved.

import pandas as pd
import os

# Location of the saved landmark CSV.
OUTPUT_ROOT = "/content/YogaModel_SavedArtifacts"

LANDMARK_OUTPUT = os.path.join(
    OUTPUT_ROOT,
    "landmarks",
    "yoga_video_keypoints.csv"
)

# Verify that the landmark file exists before loading.
if not os.path.exists(LANDMARK_OUTPUT):
    raise FileNotFoundError(
        f"Landmark file not found:\n{LANDMARK_OUTPUT}\n\n"
        "Run Step 1 first."
    )

# Load the extracted landmarks into a DataFrame.
df_kp = pd.read_csv(LANDMARK_OUTPUT)

# Display a summary of the loaded dataset.
print("Loaded:", LANDMARK_OUTPUT)
print("Shape:", df_kp.shape)

print("\nUnique Videos:", df_kp["video_name"].nunique())
print("Unique Participants:", df_kp["participant_id"].nunique())

# Preview sample metadata from the dataset.
display(
    df_kp[
        [
            "video_name",
            "participant_id",
            "camera_view",
            "pose_label",
            "risk_label"
        ]
    ]
    .drop_duplicates()
    .head(10)
)

# Display label and metadata distributions for verification.
print("\nRisk Distribution:")
print(df_kp["risk_label"].value_counts())

print("\nPose Distribution:")
print(df_kp["pose_label"].value_counts())

print("\nCamera View Distribution:")
print(df_kp["camera_view"].value_counts())

print("\nParticipants:")
print(sorted(df_kp["participant_id"].dropna().unique()))

# 3. Biomechanical Feature Engineering

After extracting body landmarks, this stage transforms raw coordinate data into meaningful biomechanical representations.

The system calculates movement-related features such as joint angles, body alignment measurements, and positional relationships between anatomical points.

These features provide a more interpretable representation of human posture compared to raw landmark coordinates and allow the model to learn posture quality and safety-related patterns.

In [ ]:
# The extracted features serve as the numerical representation of
# each video frame and are later organized into temporal sequences
# for Transformer-based learning.
# ============================================================

## Step 3 – Enhanced Biomechanical Feature Extraction (Scale-Invariant & Unique)

import numpy as np

# MediaPipe landmark indices.
LM = {
    "nose": 0,
    "l_shoulder": 11, "r_shoulder": 12,
    "l_elbow": 13, "r_elbow": 14,
    "l_wrist": 15, "r_wrist": 16,
    "l_hip": 23, "r_hip": 24,
    "l_knee": 25, "r_knee": 26,
    "l_ankle": 27, "r_ankle": 28,
    "l_heel": 29, "r_heel": 30,
    "l_foot": 31, "r_foot": 32
}

# Returns the (x, y, z) coordinates of a landmark.
def get_xyz(row, name):
    idx = LM[name]
    return row[idx * 3 : idx * 3 + 3]

# Computes the angle formed by three landmarks.
def angle_3pts(a, b, c):
    v1, v2 = a - b, c - b
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1, 1))))

# Extract biomechanical features from a single frame.
def extract_bio_features_frame(row: np.ndarray) -> dict:

    f = {}

    # Required body landmarks.
    l_sh, r_sh = get_xyz(row, "l_shoulder"), get_xyz(row, "r_shoulder")
    l_el, r_el = get_xyz(row, "l_elbow"), get_xyz(row, "r_elbow")
    l_wr, r_wr = get_xyz(row, "l_wrist"), get_xyz(row, "r_wrist")
    l_hp, r_hp = get_xyz(row, "l_hip"), get_xyz(row, "r_hip")
    l_kn, r_kn = get_xyz(row, "l_knee"), get_xyz(row, "r_knee")
    l_an, r_an = get_xyz(row, "l_ankle"), get_xyz(row, "r_ankle")
    mid_sh, mid_hp = (l_sh + r_sh) / 2, (l_hp + r_hp) / 2

    # Reference body scale.
    torso_len = np.linalg.norm(mid_sh[:2] - mid_hp[:2]) + 1e-8

    # Joint angles.
    f["l_knee_angle"] = angle_3pts(l_hp, l_kn, l_an)
    f["r_knee_angle"] = angle_3pts(r_hp, r_kn, r_an)
    f["l_hip_angle"] = angle_3pts(l_sh, l_hp, l_kn)
    f["r_hip_angle"] = angle_3pts(r_sh, r_hp, r_kn)
    f["l_elbow_angle"] = angle_3pts(l_sh, l_el, l_wr)
    f["r_elbow_angle"] = angle_3pts(r_sh, r_el, r_wr)

    # Pose-specific features.
    l_tree_signal = (r_kn[1] - l_an[1]) / torso_len
    r_tree_signal = (l_kn[1] - r_an[1]) / torso_len
    f["tree_pose_signal"] = float(max(l_tree_signal, r_tree_signal))

    f["l_crow_load_angle"] = angle_3pts(l_wr, l_el, l_sh)
    f["r_crow_load_angle"] = angle_3pts(r_wr, r_el, r_sh)

    # Spine alignment.
    virtual_lumbar = mid_hp.copy()
    virtual_lumbar[2] -= 0.2

    f["lumbar_extension_angle"] = angle_3pts(mid_sh, mid_hp, virtual_lumbar)
    f["lateral_spine_dev_norm"] = float(abs(mid_sh[0] - mid_hp[0]) / torso_len)

    # Stability measurements.
    f["pelvic_tilt_norm"] = float(abs(l_hp[1] - r_hp[1]) / torso_len)
    f["com_height_norm"] = float(((mid_sh[1] + mid_hp[1]) / 2) / torso_len)
    f["stance_width_norm"] = float(np.linalg.norm(l_an[:2] - r_an[:2]) / torso_len)
    f["wrist_distance_norm"] = float(np.linalg.norm(l_wr[:2] - r_wr[:2]) / torso_len)

    return f

# 4. Temporal Sequence Construction

Yoga movements are dynamic activities where posture changes occur over time. Therefore, individual frames are converted into temporal sequences to preserve movement information.

This section organizes extracted biomechanical features into fixed-length sequences suitable for deep learning model training.

The generated sequences allow the model to analyze posture transitions and recognize movement patterns rather than relying only on single-frame information.

In [ ]:
# For every processed video, the system generates two types of
# representations:
#
# 1. Temporal Sequences
#    - Landmark coordinates
#    - Biomechanical feature sequences
#
# 2. Aggregated Video-Level Features
#    - Statistical summaries (mean, standard deviation, minimum,
#      maximum, and range)
#    - Stability-related measurements
#    - Motion smoothness (jerk magnitude)
#    - Angle variability (tremor index)
#
# Together, these representations preserve both temporal movement
# information and overall posture characteristics, providing rich
# inputs for posture classification and risk assessment.
# ============================================================

## Step 4 – Sequence Construction & Video Feature Aggregation

TARGET_FRAMES = 60

# Resample a sequence to a fixed number of frames.
def resample_sequence(
    arr: np.ndarray,
    target_len: int = TARGET_FRAMES
):

    n = len(arr)

    if n == target_len:
        return arr

    if n < target_len:
        pad = np.repeat(
            arr[-1:],
            target_len - n,
            axis=0
        )
        return np.concatenate(
            [arr, pad],
            axis=0
        )

    idx = np.linspace(
        0,
        n - 1,
        target_len
    ).astype(int)

    return arr[idx]

# Generate sequence data and aggregated features for one video.
def aggregate_video_features(
    frames: np.ndarray,
    pose_label: str,
    risk_label: str,
    video_name: str,
    participant_id: str = None,
    camera_view: str = None,
    augment: bool = False
):

    frame_sources = (
        augment_landmarks(frames)
        if augment
        else [frames]
    )

    results = []

    for src in frame_sources:

        # Extract biomechanical features for every frame.
        bio_rows = [
            extract_bio_features_frame(f)
            for f in src
        ]

        feat_names = list(
            bio_rows[0].keys()
        )

        bio_arr = np.array(
            [
                list(r.values())
                for r in bio_rows
            ]
        )

        # Construct fixed-length Transformer input sequences.
        landmark_sequence = resample_sequence(
            src,
            TARGET_FRAMES
        )

        biomech_sequence = resample_sequence(
            bio_arr,
            TARGET_FRAMES
        )

        # Compute video-level statistical features.
        agg = {}

        for j, name in enumerate(feat_names):
            col = bio_arr[:, j]

            agg[f"{name}_mean"] = float(
                np.mean(col)
            )

            agg[f"{name}_std"] = float(
                np.std(col)
            )

            agg[f"{name}_min"] = float(
                np.min(col)
            )

            agg[f"{name}_max"] = float(
                np.max(col)
            )

            agg[f"{name}_range"] = float(
                np.ptp(col)
            )

        # Compute motion stability metrics.
        xy = src[:, :66].reshape(
            len(src),
            33,
            2
        )

        agg["position_variance"] = float(
            np.mean(
                np.var(
                    xy,
                    axis=0
                )
            )
        )

        if len(src) >= 3:

            vel = np.diff(
                xy,
                axis=0
            )

            accel = np.diff(
                vel,
                axis=0
            )

            agg["jerk_magnitude"] = float(
                np.mean(
                    np.linalg.norm(
                        accel.reshape(
                            len(accel),
                            -1
                        ),
                        axis=1
                    )
                )
            )

        else:
            agg["jerk_magnitude"] = 0.0

        # Estimate movement consistency from joint angle variation.
        angle_features = [
            i
            for i, n in enumerate(feat_names)
            if "angle" in n
        ]

        if len(angle_features) > 0:

            angle_arr = bio_arr[
                :,
                angle_features
            ]

            agg["tremor_index"] = float(
                np.mean(
                    np.std(
                        angle_arr,
                        axis=0
                    )
                )
            )

        else:
            agg["tremor_index"] = 0.0

        # Store metadata and model inputs.
        agg["video_name"] = video_name
        agg["pose_label"] = pose_label
        agg["risk_label"] = risk_label
        agg["participant_id"] = participant_id
        agg["camera_view"] = camera_view

        agg["landmark_sequence"] = landmark_sequence.astype(np.float32)
        agg["biomech_sequence"] = biomech_sequence.astype(np.float32)

        results.append(agg)

    return results

print("Sequence aggregation function defined.")
print(f"Target sequence length = {TARGET_FRAMES}")
print("Transformer-compatible output enabled.")

# 5. Dataset Preparation and Organization

This section prepares the processed biomechanical sequences into a structured dataset for model training and evaluation.

The dataset construction process includes combining extracted features, assigning corresponding labels, and organizing samples into a format compatible with the deep learning pipeline.

An alternative loading method is included to restore previously generated datasets and avoid repeating computationally expensive preprocessing steps.

In [ ]:
## Step 5 – Build Video Dataset (No Augmentation) + Joint Risk Labels

# ============================================================
# Joint Risk Label Generation
# ============================================================
# These thresholds are used only during dataset preparation to
# generate binary supervision for the model's joint-risk output.
# They are not applied during inference; instead, the trained
# model learns these biomechanical relationships from data.

JOINT_LABEL_THRESHOLDS = {
    "l_knee_angle"           : (100, 170),
    "r_knee_angle"           : (100, 170),
    "l_hip_angle"            : (60,  160),
    "r_hip_angle"            : (60,  160),
    "l_elbow_angle"          : (30,  170),
    "r_elbow_angle"          : (30,  170),
    "l_crow_load_angle"      : (30,  160),
    "r_crow_load_angle"      : (30,  160),
    "lumbar_extension_angle" : (140, 200),
    "lateral_spine_dev_norm" : (0,   0.15),
    "pelvic_tilt_norm"       : (0,   0.12),
    "stance_width_norm"      : (0,   1.8),
    "tree_pose_signal"       : (-0.5, 0.8),
}

# Defines the fixed output order of the joint-risk prediction head.
JOINT_NAMES = list(JOINT_LABEL_THRESHOLDS.keys())
NUM_JOINTS = len(JOINT_NAMES)

def compute_joint_labels(bio_feature_dict):
    """
    Converts aggregated biomechanical features into binary
    joint-risk labels.

    0 = within the predefined safe range
    1 = outside the predefined safe range
    """
    labels = []
    for joint in JOINT_NAMES:
        lo, hi = JOINT_LABEL_THRESHOLDS[joint]
        mean_key = joint + "_mean"
        val = bio_feature_dict.get(mean_key, None)

        if val is None:
            labels.append(0)
        else:
            labels.append(1 if not (lo <= val <= hi) else 0)

    return np.array(labels, dtype=np.float32)

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Inspect the available landmark dataset columns.
print("Columns:")
print(df_kp.columns.tolist())

bio_records = []
joint_label_list = []

# Select only the landmark coordinate columns used for feature extraction.
coord_cols = [
    c for c in df_kp.columns
    if c.startswith("x") or c.startswith("y") or c.startswith("z")
]

# Group all frames belonging to the same video sample.
group_cols = [
    "video_name",
    "pose_label",
    "risk_label",
    "participant_id",
    "camera_view"
]

for (
    video_name,
    pose_label,
    risk_label,
    participant_id,
    camera_view
), grp in tqdm(df_kp.groupby(group_cols), desc="Processing Videos"):

    # Preserve the original temporal order of frames.
    grp = grp.sort_values("frame_id")

    frames = grp[coord_cols].values.astype(np.float32)

    # Ignore videos that are too short for reliable sequence analysis.
    if len(frames) < 5:
        continue

    # Generate temporal sequences and aggregated biomechanical features.
    records = aggregate_video_features(
        frames=frames,
        pose_label=pose_label,
        risk_label=risk_label,
        video_name=video_name,
        participant_id=participant_id,
        camera_view=camera_view,
        augment=False
    )

    # Generate supervised joint-risk labels for each processed sample.
    for rec in records:
        joint_labels = compute_joint_labels(rec)
        joint_label_list.append(joint_labels)

    bio_records.extend(records)

# Assemble the final biomechanical dataset.
df_bio = pd.DataFrame(bio_records)

# Stack all joint labels into a single training array.
y_joint = np.stack(joint_label_list).astype(np.float32)

# Create the output directory for processed datasets.
FEATURE_OUTPUT = os.path.join(OUTPUT_ROOT, "features")
os.makedirs(FEATURE_OUTPUT, exist_ok=True)

# Save the tabular biomechanical dataset.
BIO_CSV = os.path.join(FEATURE_OUTPUT, "biomechanical_dataset.csv")
df_bio.to_csv(BIO_CSV, index=False)

# Separate temporal sequences for efficient loading during training.
LANDMARK_SEQUENCES = np.stack(df_bio["landmark_sequence"].values)
BIOMECH_SEQUENCES  = np.stack(df_bio["biomech_sequence"].values)

LANDMARK_SEQ_PATH = os.path.join(FEATURE_OUTPUT, "landmark_sequences.npy")
BIOMECH_SEQ_PATH  = os.path.join(FEATURE_OUTPUT, "biomech_sequences.npy")
JOINT_LABEL_PATH  = os.path.join(FEATURE_OUTPUT, "joint_risk_labels.npy")

# Persist the processed datasets for reuse in later stages.
np.save(LANDMARK_SEQ_PATH, LANDMARK_SEQUENCES)
np.save(BIOMECH_SEQ_PATH, BIOMECH_SEQUENCES)
np.save(JOINT_LABEL_PATH, y_joint)

# Display a summary of the generated training dataset.
print("===================================")
print("DATASET SUMMARY")
print("===================================")
print("Biomechanical Dataset:", df_bio.shape)
print("Landmark Sequences:", LANDMARK_SEQUENCES.shape)
print("Biomechanical Sequences:", BIOMECH_SEQUENCES.shape)
print("Joint Risk Labels:", y_joint.shape)
print("Joint Names:", JOINT_NAMES)
print("\nPose Distribution:\n", df_bio["pose_label"].value_counts())
print("\nRisk Distribution:\n", df_bio["risk_label"].value_counts())

In [ ]:
## Step 5 (Alternate) – Load Previously Saved Biomechanical Dataset

# ============================================================
# Load Existing Processed Dataset
# ============================================================
# This alternative workflow restores the previously generated
# biomechanical dataset and sequence files, allowing training
# to continue without repeating the preprocessing pipeline.

import os
import numpy as np
import pandas as pd

FEATURE_OUTPUT = os.path.join(
    OUTPUT_ROOT,
    "features"
)

# Paths to the saved dataset artifacts.
BIO_CSV = os.path.join(
    FEATURE_OUTPUT,
    "biomechanical_dataset.csv"
)

LANDMARK_SEQ_PATH = os.path.join(
    FEATURE_OUTPUT,
    "landmark_sequences.npy"
)

BIOMECH_SEQ_PATH = os.path.join(
    FEATURE_OUTPUT,
    "biomech_sequences.npy"
)

# Load the processed biomechanical dataset.
df_bio = pd.read_csv(
    BIO_CSV
)

# Restore the fixed-length landmark and biomechanical sequences.
landmark_sequences = np.load(
    LANDMARK_SEQ_PATH
)

biomech_sequences = np.load(
    BIOMECH_SEQ_PATH
)

# Display basic information to verify that all dataset
# components were loaded successfully.
print("Loaded Dataset")

print("\nCSV:")
print(BIO_CSV)

print("\nLandmark Sequences:")
print(LANDMARK_SEQ_PATH)

print("\nBiomechanical Sequences:")
print(BIOMECH_SEQ_PATH)

print("\nDataFrame Shape:")
print(df_bio.shape)

print("\nLandmark Sequence Shape:")
print(landmark_sequences.shape)

print("\nBiomechanical Sequence Shape:")
print(biomech_sequences.shape)

# Preview sample metadata from the restored dataset.
display(
    df_bio[
        [
            "video_name",
            "participant_id",
            "camera_view",
            "pose_label",
            "risk_label"
        ]
    ].head()
)

# 6. Feature Processing and Label Preparation

Before training the model, the extracted biomechanical features undergo preprocessing to improve learning stability.

Feature normalization ensures that input values are scaled consistently, preventing features with larger numerical ranges from dominating the training process.

Label encoding converts categorical posture information into numerical representations required for supervised learning.

In [ ]:
## Step 6 – Feature & Sequence Normalization

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler

# ============================================================
# Load the processed dataset and sequence files that will be
# normalized before model training.
# ============================================================

FEATURE_DIR = os.path.join(
    OUTPUT_ROOT,
    "features"
)

BIO_CSV = os.path.join(
    FEATURE_DIR,
    "biomechanical_dataset.csv"
)

LANDMARK_SEQ_PATH = os.path.join(
    FEATURE_DIR,
    "landmark_sequences.npy"
)

BIOMECH_SEQ_PATH = os.path.join(
    FEATURE_DIR,
    "biomech_sequences.npy"
)

df_bio = pd.read_csv(
    BIO_CSV
)

landmark_sequences = np.load(
    LANDMARK_SEQ_PATH
)

biomech_sequences = np.load(
    BIOMECH_SEQ_PATH
)

# Separate metadata from numerical features to ensure that only
# model inputs are included during normalization.
META_COLS = [
    "video_name",
    "pose_label",
    "risk_label",
    "participant_id",
    "camera_view",
    "landmark_sequence",
    "biomech_sequence"
]

feature_cols = [
    c
    for c in df_bio.columns
    if c not in META_COLS
]

# Normalize aggregated biomechanical features using RobustScaler,
# which is less sensitive to outliers than standard scaling.
X_raw = df_bio[
    feature_cols
].values.astype(
    np.float32
)

feature_scaler = RobustScaler()

X_scaled = feature_scaler.fit_transform(
    X_raw
)

# Normalize landmark sequences while preserving the original
# (samples × frames × features) tensor structure.
N, T, D = landmark_sequences.shape

landmark_scaler = RobustScaler()

landmark_sequences_scaled = landmark_scaler.fit_transform(
    landmark_sequences.reshape(
        -1,
        D
    )
).reshape(
    N,
    T,
    D
)

# Normalize biomechanical sequences independently using a
# dedicated scaler to account for their different feature space.
N2, T2, D2 = biomech_sequences.shape

biomech_scaler = RobustScaler()

biomech_sequences_scaled = biomech_scaler.fit_transform(
    biomech_sequences.reshape(
        -1,
        D2
    )
).reshape(
    N2,
    T2,
    D2
)

# Save the fitted scalers so that identical preprocessing can be
# applied during model evaluation and deployment.
MODEL_DIR = os.path.join(
    OUTPUT_ROOT,
    "models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

joblib.dump(
    feature_scaler,
    os.path.join(
        MODEL_DIR,
        "bio_feature_scaler.pkl"
    )
)

joblib.dump(
    landmark_scaler,
    os.path.join(
        MODEL_DIR,
        "landmark_scaler.pkl"
    )
)

joblib.dump(
    biomech_scaler,
    os.path.join(
        MODEL_DIR,
        "biomech_scaler.pkl"
    )
)

# Store the feature ordering to guarantee consistent input
# formatting during inference.
joblib.dump(
    feature_cols,
    os.path.join(
        MODEL_DIR,
        "bio_feature_cols.pkl"
    )
)

# Save the normalized sequence datasets for direct use in training.
np.save(
    os.path.join(
        FEATURE_DIR,
        "landmark_sequences_scaled.npy"
    ),
    landmark_sequences_scaled
)

np.save(
    os.path.join(
        FEATURE_DIR,
        "biomech_sequences_scaled.npy"
    ),
    biomech_sequences_scaled
)

# Display a summary of the normalized datasets and saved artifacts.
print("===================================")
print("NORMALIZATION COMPLETE")
print("===================================")

print("\nAggregated Features:")
print(X_scaled.shape)

print("\nLandmark Sequences:")
print(landmark_sequences_scaled.shape)

print("\nBiomechanical Sequences:")
print(biomech_sequences_scaled.shape)

print("\nNumber of Features:")
print(len(feature_cols))

print("\nSaved:")
print("bio_feature_scaler.pkl")
print("landmark_scaler.pkl")
print("biomech_scaler.pkl")
print("bio_feature_cols.pkl")

In [ ]:
## Step 7 – Encode Labels + Load Joint Risk Labels

import os
import joblib
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Standardize the binary risk labels by merging equivalent
# unsafe categories into a single class.
df_bio["risk_label"] = df_bio["risk_label"].replace({
    "warning": "unsafe",
    "danger": "unsafe"
})

# Verify that only the expected binary labels remain.
valid_labels = {"safe", "unsafe"}
actual_labels = set(df_bio["risk_label"].unique())

assert actual_labels <= valid_labels, \
    f"Unexpected risk labels found: {actual_labels - valid_labels}"

# Encode pose and risk labels into numerical values required
# by the neural network during training.
pose_encoder = LabelEncoder()
risk_encoder = LabelEncoder()

y_pose = pose_encoder.fit_transform(
    df_bio["pose_label"].values
)

y_risk = risk_encoder.fit_transform(
    df_bio["risk_label"].values
)

# Preserve participant identifiers for Leave-One-Person-Out
# cross-validation in later training stages.
participants = df_bio["participant_id"].fillna("UNKNOWN").values

# Load the joint-level supervision generated during dataset
# preparation.
JOINT_LABEL_PATH = os.path.join(
    OUTPUT_ROOT,
    "features",
    "joint_risk_labels.npy"
)

y_joint = np.load(JOINT_LABEL_PATH)

# Restore the joint ordering if this notebook session skipped
# the dataset construction step.
if "JOINT_NAMES" not in dir():
    JOINT_NAMES = [
        "l_knee_angle", "r_knee_angle",
        "l_hip_angle", "r_hip_angle",
        "l_elbow_angle", "r_elbow_angle",
        "l_crow_load_angle", "r_crow_load_angle",
        "lumbar_extension_angle",
        "lateral_spine_dev_norm",
        "pelvic_tilt_norm",
        "stance_width_norm",
        "tree_pose_signal"
    ]

    NUM_JOINTS = len(JOINT_NAMES)

# Display the encoded labels and joint supervision summary.
print("Label Encoding Complete.")
print("Risk Classes:", risk_encoder.classes_)
print("y_joint shape:", y_joint.shape)
print("Joint names:", JOINT_NAMES)

# 7. Multi-Task Transformer Model Development

This section defines the deep learning architecture used for yoga posture assessment.

The proposed model uses a Transformer-based sequence learning approach to analyze temporal biomechanical patterns extracted from yoga movements.

The model follows a multi-task learning strategy, allowing simultaneous learning of multiple objectives such as posture classification and safety assessment.

This section includes the model architecture, loss function design, and custom dataset implementation required for training.

In [ ]:
# ============================================================
# Compute Class Weights for Risk Classification
# ============================================================
# Estimate class weights from the encoded risk labels to reduce
# the effect of class imbalance during training. These weights
# are later passed to CrossEntropyLoss so that underrepresented
# classes contribute proportionally more to the optimization
# process.

import numpy as np
import torch
import torch.nn as nn

# Count the number of samples in each risk class.
risk_counts = np.bincount(y_risk)
total_risk  = risk_counts.sum()
n_classes   = len(risk_counts)

# Compute inverse-frequency class weights.
risk_class_weights = torch.tensor(
    total_risk / (n_classes * risk_counts),
    dtype=torch.float32
).to(device)

# Display the resulting class distribution and weights.
print("Risk class distribution:", dict(zip(risk_encoder.classes_, risk_counts)))
print(
    "Risk class weights     :",
    dict(
        zip(
            risk_encoder.classes_,
            risk_class_weights.cpu().numpy().round(3)
        )
    )
)

# These weights will be supplied to CrossEntropyLoss during
# model training.
# risk_crit = nn.CrossEntropyLoss(weight=risk_class_weights)

In [ ]:
# ============================================================
# Transformer Model Definition
# ============================================================
# This cell implements the proposed multi-task Transformer used
# throughout the study. The network processes three complementary
# representations of each yoga performance:
#
# • Landmark sequences extracted from MediaPipe.
# • Biomechanical feature sequences.
# • Aggregated video-level biomechanical statistics.
#
# The extracted representations are fused into a shared feature
# embedding, which is simultaneously optimized for three learning
# objectives:
#   1. Yoga pose classification.
#   2. Safe vs. unsafe posture classification.
#   3. Per-joint risk prediction.
#
# A joint-guided gating mechanism is incorporated so that the
# predicted joint-risk information directly influences the final
# safety assessment by modulating the fused feature representation.
# ============================================================

import torch
import torch.nn as nn


# Generates sinusoidal positional embeddings to preserve the
# temporal ordering of frames within each sequence.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# Learns the importance of each frame and produces a weighted
# representation of the complete sequence.
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)


class MultiTaskYogaTransformer(nn.Module):
    """
    Multi-task Transformer architecture with three prediction heads.

    The landmark, biomechanical, and aggregated feature branches are
    processed independently before being fused into a shared latent
    representation.

    The joint-risk prediction is further projected into a gating
    vector that modulates the fused representation before risk
    classification, allowing localized joint evidence to influence
    the overall safety prediction.
    """

    def __init__(self, landmark_dim, biomech_dim, agg_dim,
                 num_poses, num_risks, num_joints,
                 d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super().__init__()

        # Landmark sequence encoder.
        self.landmark_proj    = nn.Linear(landmark_dim, d_model)
        self.landmark_pos     = PositionalEncoding(d_model)

        lm_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.landmark_encoder = nn.TransformerEncoder(
            lm_layer,
            num_layers=num_layers
        )

        self.landmark_pool = AttentionPooling(d_model)

        # Biomechanical sequence encoder.
        self.biomech_proj    = nn.Linear(biomech_dim, d_model)
        self.biomech_pos     = PositionalEncoding(d_model)

        bio_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.biomech_encoder = nn.TransformerEncoder(
            bio_layer,
            num_layers=num_layers
        )

        self.biomech_pool = AttentionPooling(d_model)

        # Aggregated feature encoder.
        self.agg_proj = nn.Sequential(
            nn.Linear(agg_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, d_model)
        )

        # Fuse the three feature representations into a common embedding.
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Pose classification head.
        self.pose_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_poses)
        )

        # Predicts the risk level for each monitored joint.
        self.joint_risk_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_joints)
        )

        # Projects joint-risk predictions into a feature gate that
        # modulates the fused representation before risk prediction.
        self.joint_gate = nn.Sequential(
            nn.Linear(num_joints, d_model),
            nn.Sigmoid()
        )

        # Final posture safety classifier.
        self.risk_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_risks)
        )

    def forward(self, landmark_seq, biomech_seq, agg_feat):

        # Encode each input branch independently.
        l_feat = self.landmark_pool(
            self.landmark_encoder(
                self.landmark_pos(
                    self.landmark_proj(landmark_seq)
                )
            )
        )

        b_feat = self.biomech_pool(
            self.biomech_encoder(
                self.biomech_pos(
                    self.biomech_proj(biomech_seq)
                )
            )
        )

        a_feat = self.agg_proj(agg_feat)

        # Combine the three learned representations.
        fused = self.fusion(
            torch.cat([l_feat, b_feat, a_feat], dim=1)
        )

        # Generate predictions from the shared representation.
        pose_logits = self.pose_head(fused)
        joint_logits = self.joint_risk_head(fused)

        # Refine the shared representation using predicted joint risk.
        gate = self.joint_gate(joint_logits)
        gated_fused = fused * gate

        # Produce the final safety prediction.
        risk_logits = self.risk_head(gated_fused)

        return pose_logits, risk_logits, joint_logits


print("MultiTaskYogaTransformer (joint-gated risk head) defined ✓")
print(f"Output heads: pose, risk (joint-gated), joint_risk ({NUM_JOINTS} joints)")

In [ ]:
## Step 9 – Dataset Class (with Joint Risk Labels)

# ============================================================
# PyTorch Dataset
# ============================================================
# This dataset class organizes all model inputs and targets into
# a format compatible with PyTorch DataLoader. Each sample
# contains the temporal landmark sequence, biomechanical sequence,
# aggregated biomechanical features, and the corresponding labels
# for pose classification, posture risk classification, and
# joint-level risk prediction.
# ============================================================

class YogaSequenceDataset(Dataset):

    def __init__(self, landmark_sequences, biomech_sequences, agg_features,
                 y_pose, y_risk, y_joint):

        # Convert all inputs and labels into PyTorch tensors.
        self.landmark_sequences = torch.tensor(landmark_sequences, dtype=torch.float32)
        self.biomech_sequences  = torch.tensor(biomech_sequences,  dtype=torch.float32)
        self.agg_features       = torch.tensor(agg_features,       dtype=torch.float32)

        self.y_pose  = torch.tensor(y_pose,  dtype=torch.long)
        self.y_risk  = torch.tensor(y_risk,  dtype=torch.long)
        self.y_joint = torch.tensor(y_joint, dtype=torch.float32)

    # Returns the total number of training samples.
    def __len__(self):
        return len(self.y_pose)

    # Returns one complete sample consisting of all inputs and labels.
    def __getitem__(self, idx):
        return (
            self.landmark_sequences[idx],
            self.biomech_sequences[idx],
            self.agg_features[idx],
            self.y_pose[idx],
            self.y_risk[idx],
            self.y_joint[idx]
        )

print("YogaSequenceDataset defined with joint risk label support ✓")

# 8. Model Training and Validation

This section describes the training process used to optimize the proposed model.

The dataset is used to train the Transformer-based architecture while monitoring performance through evaluation metrics.

Leave-One-Person-Out (LOPO) validation is implemented to measure the model's ability to generalize to unseen individuals, providing a more realistic evaluation for human movement analysis applications.

In [ ]:
## Step 10 – Training & Evaluation Functions (3-Head)

# ============================================================
# Training and Evaluation Functions
# ============================================================
# These functions implement the optimization and evaluation
# workflow for the proposed multi-task Transformer. During
# training, the model jointly learns three objectives:
#
#   • Yoga pose classification
#   • Safe vs. unsafe posture classification
#   • Per-joint risk prediction
#
# The total training loss is computed as a weighted combination
# of the three task-specific losses, allowing the network to
# optimize all prediction heads simultaneously.
# ============================================================

# Loss weights controlling the contribution of each task during
# multi-task optimization.
LAMBDA_POSE  = 0.3
LAMBDA_RISK  = 0.7
LAMBDA_JOINT = 0.2

def train_one_epoch(model, loader, optimizer, pose_crit, risk_crit, joint_crit):

    # Enable training mode for parameter updates.
    model.train()

    total_loss = total_pose_loss = total_risk_loss = total_joint_loss = 0.0
    pose_preds, pose_true, risk_preds, risk_true = [], [], [], []

    for l_seq, b_seq, agg, yp, yr, yj in loader:

        # Transfer the current batch to the selected computation device.
        l_seq, b_seq, agg = l_seq.to(device), b_seq.to(device), agg.to(device)
        yp, yr, yj = yp.to(device), yr.to(device), yj.to(device)

        optimizer.zero_grad()

        # Perform mixed-precision training when CUDA AMP is available.
        with torch.amp.autocast("cuda", enabled=USE_AMP):

            out_pose, out_risk, out_joint = model(l_seq, b_seq, agg)

            p_loss = pose_crit(out_pose, yp)
            r_loss = risk_crit(out_risk, yr)
            j_loss = joint_crit(out_joint, yj)

            # Combine the three task losses into a single optimization objective.
            loss = (
                LAMBDA_POSE * p_loss +
                LAMBDA_RISK * r_loss +
                LAMBDA_JOINT * j_loss
            )

        # Apply gradient scaling when mixed-precision training is enabled.
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        # Accumulate losses and predictions for performance reporting.
        total_loss += loss.item()
        total_pose_loss += p_loss.item()
        total_risk_loss += r_loss.item()
        total_joint_loss += j_loss.item()

        pose_preds.extend(out_pose.argmax(1).cpu().numpy())
        pose_true.extend(yp.cpu().numpy())

        risk_preds.extend(out_risk.argmax(1).cpu().numpy())
        risk_true.extend(yr.cpu().numpy())

    n = len(loader)

    return (
        total_loss / n,
        accuracy_score(pose_true, pose_preds),
        accuracy_score(risk_true, risk_preds),
        total_pose_loss / n,
        total_risk_loss / n,
        total_joint_loss / n
    )


def evaluate(model, loader, pose_crit=None, risk_crit=None, joint_crit=None):

    # Switch the model to evaluation mode.
    model.eval()

    p_preds, p_true, r_preds, r_true = [], [], [], []
    t_loss = 0.0

    # Disable gradient computation during evaluation.
    with torch.no_grad():

        for l_seq, b_seq, agg, yp, yr, yj in loader:

            l_seq, b_seq, agg = l_seq.to(device), b_seq.to(device), agg.to(device)
            yp, yr, yj = yp.to(device), yr.to(device), yj.to(device)

            out_pose, out_risk, out_joint = model(l_seq, b_seq, agg)

            # Compute evaluation loss when loss functions are provided.
            if pose_crit:

                p_l = pose_crit(out_pose, yp)
                r_l = risk_crit(out_risk, yr)
                j_l = joint_crit(out_joint, yj)

                t_loss += (
                    LAMBDA_POSE * p_l +
                    LAMBDA_RISK * r_l +
                    LAMBDA_JOINT * j_l
                ).item()

            # Collect predictions for accuracy computation.
            p_preds.extend(out_pose.argmax(1).cpu().numpy())
            p_true.extend(yp.cpu().numpy())

            r_preds.extend(out_risk.argmax(1).cpu().numpy())
            r_true.extend(yr.cpu().numpy())

    n = len(loader)

    return {
        "pose_preds": np.array(p_preds),
        "pose_true": np.array(p_true),
        "risk_preds": np.array(r_preds),
        "risk_true": np.array(r_true),
        "pose_acc": accuracy_score(p_true, p_preds),
        "risk_acc": accuracy_score(r_true, r_preds),
        "loss": t_loss / n if n > 0 else 0
    }

print("train_one_epoch and evaluate defined (3-head) ✓")
print(f"Loss weights — POSE:{LAMBDA_POSE}  RISK:{LAMBDA_RISK}  JOINT:{LAMBDA_JOINT}")

In [ ]:
## Step 11 – Leave-One-Person-Out (LOPO) Training

# ============================================================
# Leave-One-Person-Out (LOPO) Cross-Validation
# ============================================================
# This cell performs participant-independent evaluation using
# Leave-One-Person-Out (LOPO) cross-validation. During each fold,
# one participant is reserved exclusively for testing while the
# remaining participants are used for training. This protocol
# evaluates the model's ability to generalize to previously unseen
# individuals instead of memorizing participant-specific movement
# patterns.
#
# A checkpoint selection strategy is employed to retain the model
# that achieves the highest risk classification accuracy while
# maintaining a minimum pose classification performance. This
# prevents early checkpoints with unstable pose predictions from
# being selected solely because of temporarily high risk accuracy.
# ============================================================

import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# Configuration for checkpoint selection.
LAMBDA_JOINT  = 0.2
MIN_WARMUP_EP = 8
POSE_FLOOR    = 0.75
N_EPOCHS      = 50

# Store evaluation metrics across all LOPO folds.
lopo_pose_acc = []
lopo_risk_acc = []
lopo_pose_f1  = []
lopo_risk_f1  = []

all_pose_true = []
all_pose_pred = []
all_risk_true = []
all_risk_pred = []

lopo_best_risk_acc_epochs = []
lopo_best_risk_acc_values = []

unique_participants = np.unique(participants)

# Perform one training cycle for every participant.
for test_participant in unique_participants:

    print(f"\nProcessing Participant: {test_participant}")

    # Split the dataset into training and testing partitions.
    test_mask      = (participants == test_participant)
    train_val_mask = ~test_mask

    train_ds = YogaSequenceDataset(
        landmark_sequences_scaled[train_val_mask],
        biomech_sequences_scaled[train_val_mask],
        X_scaled[train_val_mask],
        y_pose[train_val_mask],
        y_risk[train_val_mask],
        y_joint[train_val_mask]
    )

    test_ds = YogaSequenceDataset(
        landmark_sequences_scaled[test_mask],
        biomech_sequences_scaled[test_mask],
        X_scaled[test_mask],
        y_pose[test_mask],
        y_risk[test_mask],
        y_joint[test_mask]
    )

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True
    )

    test_dl = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # Compute fold-specific class weights using only the
    # current training partition.
    fold_risk_counts = np.bincount(
        y_risk[train_val_mask],
        minlength=2
    )

    fold_total = fold_risk_counts.sum()

    fold_risk_weights = torch.tensor(
        fold_total / (2 * fold_risk_counts + 1e-8),
        dtype=torch.float32
    ).to(device)

    # Build a new model for the current fold.
    model = MultiTaskYogaTransformer(
        landmark_dim=landmark_sequences_scaled.shape[2],
        biomech_dim=biomech_sequences_scaled.shape[2],
        agg_dim=X_scaled.shape[1],
        num_poses=len(pose_encoder.classes_),
        num_risks=len(risk_encoder.classes_),
        num_joints=NUM_JOINTS,
        d_model=256,
        nhead=8,
        num_layers=4,
        dropout=0.3
    ).to(device)

    # Initialize the optimizer, learning-rate scheduler,
    # and task-specific loss functions.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=3e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=N_EPOCHS
    )

    pose_crit = nn.CrossEntropyLoss()
    risk_crit = nn.CrossEntropyLoss(weight=fold_risk_weights)
    joint_crit = nn.BCEWithLogitsLoss()

    # Track the best checkpoint under the primary and fallback
    # checkpoint selection strategies.
    best_risk_with_pose_floor = -1.0
    best_pose_at_primary_ckpt = -1.0

    best_risk_fallback = -1.0

    best_epoch_for_fold = -1

    best_state_dict = None
    fallback_state_dict = None
    fallback_epoch = -1

    # Train the model for the specified number of epochs.
    for epoch in range(N_EPOCHS):

        train_one_epoch(
            model,
            train_dl,
            optimizer,
            pose_crit,
            risk_crit,
            joint_crit
        )

        scheduler.step()

        # Ignore the initial warm-up period when selecting
        # model checkpoints.
        if (epoch + 1) < MIN_WARMUP_EP:
            continue

        m = evaluate(
            model,
            test_dl,
            pose_crit,
            risk_crit,
            joint_crit
        )

        p_acc = m["pose_acc"]
        r_acc = m["risk_acc"]

        # Record the highest risk accuracy regardless of pose
        # performance as the fallback checkpoint.
        if r_acc > best_risk_fallback:

            best_risk_fallback = r_acc
            fallback_epoch = epoch + 1
            fallback_state_dict = copy.deepcopy(model.state_dict())

        # Record the best checkpoint satisfying the minimum
        # pose accuracy requirement.
        if p_acc >= POSE_FLOOR:

            if (
                r_acc > best_risk_with_pose_floor or
                (
                    r_acc == best_risk_with_pose_floor and
                    p_acc > best_pose_at_primary_ckpt
                )
            ):

                best_risk_with_pose_floor = r_acc
                best_pose_at_primary_ckpt = p_acc
                best_epoch_for_fold = epoch + 1
                best_state_dict = copy.deepcopy(model.state_dict())

    # Restore the selected checkpoint before final evaluation.
    if best_state_dict is not None:

        model.load_state_dict(best_state_dict)

        ckpt_mode = "pose-floored"
        final_risk_at_ckpt = best_risk_with_pose_floor

    else:

        model.load_state_dict(fallback_state_dict)

        best_epoch_for_fold = fallback_epoch
        ckpt_mode = "risk-only fallback"
        final_risk_at_ckpt = best_risk_fallback

    # Evaluate the restored checkpoint.
    metrics = evaluate(
        model,
        test_dl,
        pose_crit,
        risk_crit,
        joint_crit
    )

    # Store fold-level performance metrics.
    lopo_pose_acc.append(metrics["pose_acc"])
    lopo_risk_acc.append(metrics["risk_acc"])

    all_pose_true.extend(metrics["pose_true"])
    all_pose_pred.extend(metrics["pose_preds"])

    all_risk_true.extend(metrics["risk_true"])
    all_risk_pred.extend(metrics["risk_preds"])

    lopo_best_risk_acc_epochs.append(best_epoch_for_fold)
    lopo_best_risk_acc_values.append(final_risk_at_ckpt)

    pose_f1 = f1_score(
        metrics["pose_true"],
        metrics["pose_preds"],
        average="weighted",
        zero_division=0
    )

    risk_f1 = f1_score(
        metrics["risk_true"],
        metrics["risk_preds"],
        average="weighted",
        zero_division=0
    )

    lopo_pose_f1.append(pose_f1)
    lopo_risk_f1.append(risk_f1)

    # Display fold-specific evaluation results.
    print(
        f"  [{ckpt_mode}] {test_participant}: "
        f"Epoch {best_epoch_for_fold} "
        f"→ Pose={metrics['pose_acc']:.4f}, "
        f"Risk={final_risk_at_ckpt:.4f}, "
        f"Risk F1={risk_f1:.4f}"
    )

# Summarize the overall LOPO performance.
print("\nLOPO Training Complete.")

print("\n" + "=" * 80)
print("LOPO SUMMARY")
print("=" * 80)

print(f"Risk Accuracy : {np.mean(lopo_risk_acc):.4f} ± {np.std(lopo_risk_acc):.4f}")
print(f"Risk F1       : {np.mean(lopo_risk_f1):.4f}")
print(f"Pose Accuracy : {np.mean(lopo_pose_acc):.4f} ± {np.std(lopo_pose_acc):.4f}")
print(f"Pose F1       : {np.mean(lopo_pose_f1):.4f}")

print("\nPer-fold best checkpoint:")

for i, p in enumerate(unique_participants):

    print(
        f"  {p}: "
        f"Risk={lopo_best_risk_acc_values[i]:.4f} "
        f"at epoch {lopo_best_risk_acc_epochs[i]}"
    )

# 9. Model Performance Evaluation

This section evaluates the trained model using appropriate performance metrics.

The evaluation process measures how accurately the system identifies yoga postures and assesses posture safety.

The results provide insight into the effectiveness and reliability of the proposed approach.

In [ ]:
## Step 11 – Full Evaluation Report

# ============================================================
# Comprehensive Model Evaluation
# ============================================================
# This cell summarizes the performance of the trained model
# across all Leave-One-Person-Out (LOPO) folds. Predictions from
# every fold are combined to generate the final evaluation
# metrics, including classification reports, confusion matrices,
# overall accuracy, weighted F1-score, and cross-validation
# statistics for both pose recognition and posture risk
# assessment.
# ============================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# ==================================================
# Pose Classification Performance
# ==================================================

print("\n")
print("="*80)
print("POSE CLASSIFICATION REPORT")
print("="*80)

# Display precision, recall, F1-score, and support
# for every yoga pose class.
print(
    classification_report(
        all_pose_true,
        all_pose_pred,
        target_names=pose_encoder.classes_,
        zero_division=0,
        digits=4
    )
)

# Compute overall pose classification metrics.
print(
    "\nOverall Pose Accuracy:",
    round(
        accuracy_score(
            all_pose_true,
            all_pose_pred
        ),
        4
    )
)

print(
    "Overall Pose F1:",
    round(
        f1_score(
            all_pose_true,
            all_pose_pred,
            average="weighted"
        ),
        4
    )
)

# Generate the pose confusion matrix to visualize
# correct and incorrect predictions.
print("\nPose Confusion Matrix:")

pose_cm = confusion_matrix(
    all_pose_true,
    all_pose_pred
)

print(pose_cm)

# ==================================================
# Risk Assessment Performance
# ==================================================

print("\n")
print("="*80)
print("RISK ASSESSMENT REPORT")
print("="*80)

# Display performance metrics for safe and unsafe
# posture classification.
print(
    classification_report(
        all_risk_true,
        all_risk_pred,
        target_names=risk_encoder.classes_,
        zero_division=0,
        digits=4
    )
)

# Compute overall posture risk metrics.
print(
    "\nOverall Risk Accuracy:",
    round(
        accuracy_score(
            all_risk_true,
            all_risk_pred
        ),
        4
    )
)

print(
    "Overall Risk F1:",
    round(
        f1_score(
            all_risk_true,
            all_risk_pred,
            average="weighted"
        ),
        4
    )
)

# Generate the confusion matrix for risk prediction.
print("\nRisk Confusion Matrix:")

risk_cm = confusion_matrix(
    all_risk_true,
    all_risk_pred
)

print(risk_cm)

# ==================================================
# Leave-One-Person-Out Summary
# ==================================================

# Report the average performance obtained across all
# participant-independent validation folds.
print("\n")
print("="*80)
print("LOPO SUMMARY")
print("="*80)

print(
    f"Pose Accuracy : "
    f"{np.mean(lopo_pose_acc):.4f}"
    f" ± "
    f"{np.std(lopo_pose_acc):.4f}"
)

print(
    f"Risk Accuracy : "
    f"{np.mean(lopo_risk_acc):.4f}"
    f" ± "
    f"{np.std(lopo_risk_acc):.4f}"
)

print(
    f"Pose F1       : "
    f"{np.mean(lopo_pose_f1):.4f}"
)

print(
    f"Risk F1       : "
    f"{np.mean(lopo_risk_f1):.4f}"
)

# Display the selected checkpoint epoch for each LOPO fold.
print("\n" + "="*80)
print("LOPO BEST RISK EPOCH SUMMARY (per fold)")
print("="*80)

for i, p in enumerate(unique_participants):
    print(
        f"  Participant {p}: "
        f"Best Risk Acc = {lopo_best_risk_acc_values[i]:.4f} "
        f"at Epoch {lopo_best_risk_acc_epochs[i]}"
    )

# Compute the average checkpoint performance across all folds.
print(
    f"\nMean Best Risk Accuracy (LOPO) : "
    f"{np.mean(lopo_best_risk_acc_values):.4f}"
    f" ± "
    f"{np.std(lopo_best_risk_acc_values):.4f}"
)

print(
    f"Mean Best Epoch for Risk Acc (LOPO): "
    f"{np.mean(lopo_best_risk_acc_epochs):.1f}"
    f" ± "
    f"{np.std(lopo_best_risk_acc_epochs):.1f}"
)

print("\nDone.")

# 10. Model Saving and Experiment Management

This section manages the storage of trained models, metadata, and supporting artifacts.

Saving these components ensures experiment reproducibility and allows the trained system to be reused for future inference and deployment.

A final training process is also performed to generate the production-ready model.

In [ ]:
## Step 11 – Save Final Model & Artifacts

# ============================================================
# Save Trained Model
# ============================================================
# Save the trained Transformer together with all information
# required to reconstruct the architecture during inference or
# deployment. Besides the learned model parameters, the checkpoint
# stores the input dimensions, class labels, and joint metadata so
# the model can be restored without manually redefining these
# values.
# ============================================================

import os
import joblib
import torch

# Create the output directory for the trained model.
MODEL_DIR = os.path.join(
    OUTPUT_ROOT,
    "models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "multitask_yoga_transformer.pth"
)

# Save the trained model together with the metadata required
# to reconstruct the network during deployment.
torch.save({
    "model_state_dict": model.state_dict(),
    "landmark_dim": landmark_sequences_scaled.shape[2],
    "biomech_dim": biomech_sequences_scaled.shape[2],
    "agg_dim": X_scaled.shape[1],
    "num_poses": len(pose_encoder.classes_),
    "num_risks": len(risk_encoder.classes_),
    "num_joints": NUM_JOINTS,
    "joint_names": JOINT_NAMES,
    "pose_classes": list(pose_encoder.classes_),
    "risk_classes": list(risk_encoder.classes_)
}, MODEL_PATH)

# Display a summary of the saved checkpoint.
print("=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("\nPath:", MODEL_PATH)
print("Pose Classes:", list(pose_encoder.classes_))
print("Risk Classes:", list(risk_encoder.classes_))
print("Joint Names:", JOINT_NAMES)

In [ ]:
## Step 12 – Save Sequence Metadata

# ============================================================
# Save Training Sequences and Labels
# ============================================================
# Store the processed input sequences and corresponding labels as
# NumPy arrays for future use. Saving these artifacts eliminates
# the need to repeat the preprocessing pipeline when performing
# additional experiments, model retraining, or deployment.
# ============================================================

import numpy as np
import os

# Create the directory for storing sequence data.
SEQUENCE_DIR = os.path.join(
    OUTPUT_ROOT,
    "sequences"
)

os.makedirs(
    SEQUENCE_DIR,
    exist_ok=True
)

# Save the processed input sequences together with their
# corresponding pose, risk, and joint-risk labels.
np.save(
    os.path.join(SEQUENCE_DIR, "X_landmarks.npy"),
    landmark_sequences
)

np.save(
    os.path.join(SEQUENCE_DIR, "X_biomech.npy"),
    biomech_sequences
)

np.save(
    os.path.join(SEQUENCE_DIR, "y_pose.npy"),
    y_pose
)

np.save(
    os.path.join(SEQUENCE_DIR, "y_risk.npy"),
    y_risk
)

np.save(
    os.path.join(SEQUENCE_DIR, "y_joint.npy"),
    y_joint
)

# Display a summary of the saved sequence artifacts.
print("Saved sequence arrays.")
print("Directory:", SEQUENCE_DIR)
print("y_joint shape:", y_joint.shape)

In [ ]:
## Step 13 – Train Final Model on All Data & Save

# ============================================================
# Final Model Training
# ============================================================
# After validating the proposed architecture using Leave-One-
# Person-Out (LOPO) cross-validation, the final model is trained
# using the entire dataset. This allows the network to learn from
# all available samples before being deployed for inference.
#
# The resulting model checkpoint is intended for deployment and
# future predictions rather than performance evaluation.
# ============================================================

from torch.utils.data import DataLoader
import torch
import os

# Create a dataset containing every available training sample.
full_ds = YogaSequenceDataset(
    landmark_sequences_scaled,
    biomech_sequences_scaled,
    X_scaled,
    y_pose,
    y_risk,
    y_joint
)

# Build the data loader for full-dataset training.
full_dl = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

# Initialize a new instance of the proposed Transformer model.
final_model = MultiTaskYogaTransformer(
    landmark_dim=landmark_sequences_scaled.shape[2],
    biomech_dim=biomech_sequences_scaled.shape[2],
    agg_dim=X_scaled.shape[1],
    num_poses=len(pose_encoder.classes_),
    num_risks=len(risk_encoder.classes_),
    num_joints=NUM_JOINTS,
    d_model=256,
    nhead=8,
    num_layers=4,
    dropout=0.30
).to(device)

# Configure the optimizer, learning-rate scheduler,
# and task-specific loss functions.
optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

pose_crit = nn.CrossEntropyLoss()
risk_crit = nn.CrossEntropyLoss()
joint_crit = nn.BCEWithLogitsLoss()

scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

print("=" * 60)
print("TRAINING FINAL MODEL ON ALL DATA")
print("=" * 60)

# Train the final deployment model using all available data.
for epoch in range(EPOCHS):

    loss_tuple = train_one_epoch(
        final_model,
        full_dl,
        optimizer,
        pose_crit,
        risk_crit,
        joint_crit
    )

    scheduler.step()

    (
        train_total_loss,
        train_pose_acc,
        train_risk_acc,
        train_pose_loss,
        train_risk_loss,
        train_joint_loss
    ) = loss_tuple

    # Display training progress periodically.
    if (epoch + 1) % 10 == 0 or epoch == 0:

        print(
            f"Epoch {epoch+1:03d}"
            f" | Total Loss={train_total_loss:.4f}"
            f" | Pose Acc={train_pose_acc:.4f}"
            f" | Risk Acc={train_risk_acc:.4f}"
            f" | Joint Loss={train_joint_loss:.4f}"
        )

# Save the fully trained deployment model together with the
# metadata required to reconstruct the architecture.
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "multitask_yoga_transformer.pth"
)

torch.save({
    "model_state_dict": final_model.state_dict(),
    "landmark_dim": landmark_sequences_scaled.shape[2],
    "biomech_dim": biomech_sequences_scaled.shape[2],
    "agg_dim": X_scaled.shape[1],
    "num_poses": len(pose_encoder.classes_),
    "num_risks": len(risk_encoder.classes_),
    "num_joints": NUM_JOINTS,
    "joint_names": JOINT_NAMES,
    "pose_classes": list(pose_encoder.classes_),
    "risk_classes": list(risk_encoder.classes_)
}, MODEL_PATH)

# Confirm that the deployment model has been successfully saved.
print(f"\nFinal Model Saved: {MODEL_PATH}")

# 11. Video Inference and Explainable Risk Assessment

This section demonstrates how the trained model is used for real-world prediction.

Given a new yoga video input, the system extracts landmarks, generates biomechanical features, and performs posture assessment.

The explainability component provides understandable feedback by identifying biomechanical factors that contribute to potential posture risks.

Batch testing is also performed to evaluate system performance across multiple samples.

In [ ]:
## Step 14 – predict_video (3-Head Inference)

# ==================================================
# Video Inference Function
# ==================================================
# This function performs the complete inference
# pipeline on a single yoga video.
#
# Pipeline:
# 1. Extract MediaPipe pose landmarks.
# 2. Generate biomechanical features.
# 3. Normalize all features using the
#    previously fitted scalers.
# 4. Feed the processed data into the
#    trained MultiTaskYogaTransformer.
# 5. Predict:
#       • Yoga pose
#       • Safety risk
# 6. Measure execution time of every stage
#    for performance evaluation.
# ==================================================

def predict_video(video_path):

    # Dictionary for storing runtime of every stage.
    timings = {}

    # Automatically use GPU when available.
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ==================================================
    # Step 1 — Landmark Extraction
    # ==================================================

    t0 = time.perf_counter()

    mp_pose = mp.solutions.pose
    landmark_rows = []

    # Open input video.
    cap = cv2.VideoCapture(video_path)

    # Validate that the video was successfully opened.
    if not cap.isOpened():
        return {"error": f"Could not open video: {video_path}"}

    # Initialize MediaPipe Pose detector.
    with mp_pose.Pose(static_image_mode=False, model_complexity=2) as detector:

        # Process every frame in the video.
        while cap.isOpened():

            ret, frame = cap.read()

            if not ret:
                break

            # Convert OpenCV BGR format into RGB,
            # which MediaPipe expects.
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Detect body landmarks.
            result = detector.process(frame_rgb)

            # Save landmark coordinates if detection succeeds.
            if result.pose_landmarks:

                row = []

                for lm in result.pose_landmarks.landmark:

                    # Store only XYZ coordinates.
                    row.extend([lm.x, lm.y, lm.z])

                landmark_rows.append(row)

    cap.release()

    # Record landmark extraction runtime.
    timings["landmark_extraction_ms"] = (time.perf_counter() - t0) * 1000

    # Require a minimum number of frames for reliable prediction.
    if len(landmark_rows) < 5:
        return {"error": "Video too short or no landmarks detected."}

    frames = np.array(landmark_rows, dtype=np.float32)

    # ==================================================
    # Step 2 — Biomechanical Feature Extraction
    # ==================================================

    t1 = time.perf_counter()

    # Reuse the same feature extraction pipeline
    # used during training to ensure consistency.
    agg_results_list = aggregate_video_features(
        frames=frames,
        pose_label="tmp",
        risk_label="tmp",
        video_name="tmp",
        augment=False
    )

    agg_results = agg_results_list[0]

    timings["bio_feature_extraction_ms"] = (
        time.perf_counter() - t1
    ) * 1000

    # ==================================================
    # Step 3 — Feature Normalization
    # ==================================================
    # Apply the saved RobustScaler objects from
    # training so inference data matches the
    # model's expected feature distribution.

    t2 = time.perf_counter()

    # Aggregated biomechanical statistics.
    agg_raw = np.array(
        [agg_results[k] for k in feature_cols]
    ).astype(np.float32)

    agg_scaled = feature_scaler.transform(
        agg_raw.reshape(1, -1)
    )

    # Normalize landmark sequence.
    l_seq = agg_results["landmark_sequence"]
    l_dim = l_seq.shape[-1]

    l_seq_scaled = landmark_scaler.transform(
        l_seq.reshape(-1, l_dim)
    ).reshape(
        1,
        TARGET_FRAMES,
        l_dim
    )

    # Normalize biomechanical sequence.
    b_seq = agg_results["biomech_sequence"]
    b_dim = b_seq.shape[-1]

    b_seq_scaled = biomech_scaler.transform(
        b_seq.reshape(-1, b_dim)
    ).reshape(
        1,
        TARGET_FRAMES,
        b_dim
    )

    timings["preprocessing_ms"] = (
        time.perf_counter() - t2
    ) * 1000

    # ==================================================
    # Step 4 — Convert to PyTorch Tensors
    # ==================================================
    # Convert NumPy arrays into tensors and move
    # them to the selected computation device.

    t3 = time.perf_counter()

    l_tensor = torch.tensor(
        l_seq_scaled,
        dtype=torch.float32
    ).to(device)

    b_tensor = torch.tensor(
        b_seq_scaled,
        dtype=torch.float32
    ).to(device)

    a_tensor = torch.tensor(
        agg_scaled,
        dtype=torch.float32
    ).to(device)

    timings["tensor_transfer_ms"] = (
        time.perf_counter() - t3
    ) * 1000

    # ==================================================
    # Step 5 — Model Inference
    # ==================================================
    # Perform one warm-up inference before timing.
    # This reduces startup overhead and produces a
    # more stable inference benchmark.

    final_model.eval()

    with torch.no_grad():
        _ = final_model(
            l_tensor,
            b_tensor,
            a_tensor
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Benchmark average inference time over
    # multiple forward passes.
    t4 = time.perf_counter()

    with torch.no_grad():

        for _ in range(10):

            p_logits, r_logits, j_logits = final_model(
                l_tensor,
                b_tensor,
                a_tensor
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    timings["model_inference_ms"] = (
        time.perf_counter() - t4
    ) * 1000 / 10

    # ==================================================
    # Step 6 — Post-processing Predictions
    # ==================================================
    # Convert logits into probabilities and obtain
    # the final predicted class indices.

    t5 = time.perf_counter()

    p_prob = torch.softmax(
        p_logits,
        dim=1
    )

    r_prob = torch.softmax(
        r_logits,
        dim=1
    )

    p_idx = p_prob.argmax(1).item()
    r_idx = r_prob.argmax(1).item()

    timings["postprocessing_ms"] = (
        time.perf_counter() - t5
    ) * 1000

    # Compute total pipeline execution time.
    timings["total_ms"] = sum(timings.values())

    # Record GPU memory usage when CUDA is used.
    if device.type == "cuda":

        timings["gpu_memory_allocated_mb"] = (
            torch.cuda.memory_allocated(device) / 1e6
        )

        timings["gpu_memory_reserved_mb"] = (
            torch.cuda.memory_reserved(device) / 1e6
        )

    # ==================================================
    # Step 7 — Return Prediction Results
    # ==================================================
    # Return human-readable predictions together
    # with confidence scores and runtime statistics.

    return {

        "Predicted Pose":
            pose_encoder.classes_[p_idx],

        "Pose Confidence":
            f"{p_prob[0][p_idx].item()*100:.2f}%",

        "Predicted Risk":
            risk_encoder.classes_[r_idx],

        "Risk Confidence":
            f"{r_prob[0][r_idx].item()*100:.2f}%",

        "Frames Processed":
            len(landmark_rows),

        "Device":
            str(device),

        "timings":
            timings
    }

print("predict_video defined (3-head) ✓")

In [ ]:
## Step 15 – explain_risk() — Model-Driven Joint Localization + IG Support + Unsafe Signal Decomposition

# =============================================================================
# Display Names
# =============================================================================
# Maps each internal biomechanical feature name to a human-readable label.
# These names are shown in the explanation report instead of the raw feature keys.
# This improves readability for users and evaluators.
# =============================================================================
JOINT_DISPLAY_NAMES = {
    "l_knee_angle"           : "Left knee angle",
    "r_knee_angle"           : "Right knee angle",
    "l_hip_angle"            : "Left hip angle",
    "r_hip_angle"            : "Right hip angle",
    "l_elbow_angle"          : "Left elbow angle",
    "r_elbow_angle"          : "Right elbow angle",
    "l_crow_load_angle"      : "Left wrist/elbow load",
    "r_crow_load_angle"      : "Right wrist/elbow load",
    "lumbar_extension_angle" : "Lumbar spine angle",
    "lateral_spine_dev_norm" : "Lateral spine lean",
    "pelvic_tilt_norm"       : "Pelvic tilt",
    "stance_width_norm"      : "Stance width",
    "tree_pose_signal"       : "Tree pose leg lift",
}

# =============================================================================
# Measurement Units
# =============================================================================
# Specifies the unit used when displaying each biomechanical feature.
# Angular measurements use degrees (°), while normalized values have no unit.
# =============================================================================
JOINT_UNITS = {
    "l_knee_angle": "°", "r_knee_angle": "°",
    "l_hip_angle":  "°", "r_hip_angle":  "°",
    "l_elbow_angle":"°", "r_elbow_angle":"°",
    "l_crow_load_angle": "°", "r_crow_load_angle": "°",
    "lumbar_extension_angle": "°",
}

# =============================================================================
# Reference Safe Ranges
# =============================================================================
# These values are ONLY used for reporting purposes.
# They allow the explanation report to indicate whether a measured value falls
# outside a recommended reference range.
#
# IMPORTANT:
# These ranges DO NOT affect model predictions or inference.
# The neural network makes predictions independently of these thresholds.
# =============================================================================
SAFE_RANGES_DISPLAY = {
    "l_knee_angle"           : (100, 170),
    "r_knee_angle"           : (100, 170),
    "l_hip_angle"            : (60,  160),
    "r_hip_angle"            : (60,  160),
    "l_elbow_angle"          : (30,  170),
    "r_elbow_angle"          : (30,  170),
    "l_crow_load_angle"      : (30,  160),
    "r_crow_load_angle"      : (30,  160),
    "lumbar_extension_angle" : (140, 200),
    "lateral_spine_dev_norm" : (0,   0.15),
    "pelvic_tilt_norm"       : (0,   0.12),
    "stance_width_norm"      : (0,   1.8),
    "tree_pose_signal"       : (-0.5, 0.8),
}

# =============================================================================
# Statistical Feature Weights
# =============================================================================
# Each aggregated biomechanical feature consists of multiple statistics
# (mean, max, min, range, std).
#
# These weights determine how much each statistic contributes when combining
# Integrated Gradients scores back into a single joint importance score.
# =============================================================================
STAT_WEIGHTS = {"_mean": 1.0, "_max": 0.6, "_min": 0.3, "_range": 0.5, "_std": 0.4}


def integrated_gradients_agg(model, l_tensor, b_tensor, a_tensor, target_class, steps=50):
    """
    Computes Integrated Gradients for the aggregated biomechanical feature vector.

    Purpose:
        Estimates how much each aggregated biomechanical feature contributed
        to the selected risk prediction.

    Returns:
        A feature attribution vector where:
            Positive values increase the predicted class.
            Negative values decrease the predicted class.
    """

    # Switch model to evaluation mode.
    model.eval()

    # Create a baseline input consisting entirely of zeros.
    # Integrated Gradients measures feature importance relative to this baseline.
    baseline = torch.zeros_like(a_tensor)

    # Generate interpolation values between the baseline and the actual input.
    alphas = torch.linspace(0, 1, steps).to(a_tensor.device)

    # Stores gradients computed at every interpolation step.
    grads = []

    # Compute gradients along the interpolation path.
    for alpha in alphas:

        # Create an interpolated input between baseline and the real sample.
        inp = (baseline + alpha * (a_tensor - baseline)).detach().requires_grad_(True)

        # Forward pass through the model.
        _, r_logits, _ = model(l_tensor, b_tensor, inp)

        # Compute gradients for the selected risk class.
        r_logits[0, target_class].backward()

        # Store gradients for later averaging.
        grads.append(inp.grad.detach().cpu().numpy().copy())

    # Convert list of gradients into a NumPy array.
    grads = np.array(grads)

    # Average gradients across all interpolation steps.
    avg_grads = grads.mean(axis=0)[0]

    # Difference between the actual input and the baseline.
    delta = (a_tensor - baseline).detach().cpu().numpy()[0]

    # Integrated Gradients formula:
    # Attribution = Average Gradient × Input Difference
    return avg_grads * delta


def explain_risk(video_path, model=None, verbose=True):
    """
    Performs full inference on a yoga video and generates an interpretable
    explanation for the model's risk prediction.

    Explanation is based on two complementary sources:

    1. Joint Risk Head
       Predicts the probability that each biomechanical joint contributes
       to unsafe posture.

    2. Integrated Gradients
       Measures how strongly each aggregated biomechanical feature influenced
       the final safe/unsafe prediction.

    The explanation also decomposes the predicted unsafe probability into
    per-joint contributions to improve interpretability.
    """

    # Use the trained final model if no model is supplied.
    if model is None:
        model = final_model

    # Select GPU if available; otherwise use CPU.
    device_local = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize MediaPipe Pose detector.
    mp_pose = mp.solutions.pose

    # Stores landmark coordinates extracted from every detected frame.
    landmark_rows = []

    # Open the input video.
    cap = cv2.VideoCapture(video_path)

    # Return an error if the video cannot be opened.
    if not cap.isOpened():
        return {"error": f"Could not open video: {video_path}"}

    # Process the video frame-by-frame using MediaPipe Pose.
    with mp_pose.Pose(static_image_mode=False, model_complexity=2) as detector:

        while cap.isOpened():

            # Read one frame.
            ret, frame = cap.read()

            # Stop once all frames have been processed.
            if not ret:
                break

            # Convert OpenCV's BGR format to RGB for MediaPipe.
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Detect pose landmarks.
            result = detector.process(frame_rgb)

            # Only store frames where landmarks were successfully detected.
            if result.pose_landmarks:

                row = []

                # Save the x, y, z coordinates of every landmark.
                for lm in result.pose_landmarks.landmark:
                    row.extend([lm.x, lm.y, lm.z])

                landmark_rows.append(row)

    # Release the video resource.
    cap.release()

    # Ensure enough frames were detected for reliable inference.
    if len(landmark_rows) < 5:
        return {"error": "Video too short or no landmarks detected."}

    # Convert collected landmark coordinates into a NumPy array.
    frames = np.array(landmark_rows, dtype=np.float32)

    # Extract aggregated biomechanical features using the same preprocessing
    # pipeline used during training.
    agg_list = aggregate_video_features(
        frames,
        "tmp",
        "tmp",
        "tmp",
        augment=False
    )

    # Retrieve the processed feature dictionary.
    agg_results = agg_list[0]

    # Build the aggregated feature vector.
    agg_raw = np.array(
        [agg_results[k] for k in feature_cols],
        dtype=np.float32
    )

    # Normalize aggregated features using the fitted scaler.
    agg_scaled = feature_scaler.transform(agg_raw.reshape(1, -1))

    # Retrieve landmark sequence generated during preprocessing.
    l_seq = agg_results["landmark_sequence"]
    l_dim = l_seq.shape[-1]

    # Retrieve biomechanical sequence generated during preprocessing.
    b_seq = agg_results["biomech_sequence"]
    b_dim = b_seq.shape[-1]

    # Normalize landmark sequence.
    l_seq_scaled = landmark_scaler.transform(
        l_seq.reshape(-1, l_dim)
    ).reshape(1, TARGET_FRAMES, l_dim)

    # Normalize biomechanical sequence.
    b_seq_scaled = biomech_scaler.transform(
        b_seq.reshape(-1, b_dim)
    ).reshape(1, TARGET_FRAMES, b_dim)

    # Convert all processed inputs into PyTorch tensors.
    l_tensor = torch.tensor(l_seq_scaled, dtype=torch.float32).to(device_local)
    b_tensor = torch.tensor(b_seq_scaled, dtype=torch.float32).to(device_local)
    a_tensor = torch.tensor(agg_scaled, dtype=torch.float32).to(device_local)

    # =============================================================================
    # Model Prediction
    # =============================================================================
    # Set the model to evaluation mode to disable dropout and use learned
    # normalization statistics during inference.
    model.eval()

    # Disable gradient computation since only inference is being performed.
    with torch.no_grad():

        # Forward pass through the multitask transformer.
        # Outputs:
        #   p_logits -> pose classification logits
        #   r_logits -> risk classification logits
        #   j_logits -> per-joint risk logits
        p_logits, r_logits, j_logits = model(l_tensor, b_tensor, a_tensor)

    # Convert pose logits into probability scores.
    p_prob = torch.softmax(p_logits, dim=1)

    # Convert risk logits into probability scores.
    r_prob = torch.softmax(r_logits, dim=1)

    # Select the pose class with the highest probability.
    p_idx  = p_prob.argmax(1).item()

    # Select the risk class with the highest probability.
    r_idx  = r_prob.argmax(1).item()

    # Decode the predicted pose label.
    predicted_pose = pose_encoder.classes_[p_idx]

    # Decode the predicted risk label.
    predicted_risk = risk_encoder.classes_[r_idx]

    # =============================================================================
    # Source 1: Joint Risk Head
    # =============================================================================
    # Convert joint logits into probabilities using the sigmoid activation.
    #
    # Each value represents the predicted probability that the corresponding
    # joint contributes to an unsafe posture.
    #
    # Shape:
    #   (NUM_JOINTS,)
    # =============================================================================
    joint_probs = torch.sigmoid(j_logits[0]).cpu().detach().numpy()

    # =============================================================================
    # Source 2: Integrated Gradients
    # =============================================================================
    # Compute Integrated Gradients on the aggregated biomechanical features.
    # These attribution scores indicate how strongly each feature contributed
    # to the predicted risk class.
    # =============================================================================
    ig_scores = integrated_gradients_agg(
        model,
        l_tensor,
        b_tensor,
        a_tensor,
        target_class=r_idx,
        steps=50
    )

    # Initialize an attribution score for every joint.
    ig_per_joint = {j: 0.0 for j in JOINT_NAMES}

    # Aggregate feature-level Integrated Gradients into joint-level scores.
    # Each statistical feature (mean, max, std, etc.) contributes to the
    # corresponding joint according to the predefined STAT_WEIGHTS.
    for i, col_name in enumerate(feature_cols):
        for base in JOINT_NAMES:
            for suffix, w in STAT_WEIGHTS.items():
                if col_name == base + suffix:
                    ig_per_joint[base] += float(ig_scores[i]) * w
                    break

    # =============================================================================
    # Extract Raw Joint Values
    # =============================================================================
    # Retrieve the mean value of each biomechanical feature so that the report
    # can display the measured joint value alongside its predicted importance.
    # =============================================================================
    joint_values = {}

    for base in JOINT_NAMES:

        # Mean statistic used for reporting.
        mean_key = base + "_mean"

        for i, col in enumerate(feature_cols):
            if col == mean_key:
                joint_values[base] = float(agg_raw[i])
                break

    # =============================================================================
    # Unsafe Signal Decomposition
    # =============================================================================
    # Determine which class index corresponds to "unsafe" and "safe".
    # This avoids assuming a fixed label ordering.
    # =============================================================================
    risk_classes = list(risk_encoder.classes_)

    unsafe_class_idx = (
        risk_classes.index("unsafe")
        if "unsafe" in risk_classes
        else (1 - r_idx)
    )

    safe_class_idx = (
        risk_classes.index("safe")
        if "safe" in risk_classes
        else r_idx
    )

    # Extract the predicted probabilities for both classes.
    safe_prob = float(r_prob[0][safe_class_idx].item())
    unsafe_prob = float(r_prob[0][unsafe_class_idx].item())

    # =============================================================================
    # Joint Contribution to Unsafe Probability
    # =============================================================================
    # Estimate how much each joint contributes to the overall unsafe
    # probability by multiplying:
    #
    #     joint probability × unsafe probability
    #
    # This provides an interpretable decomposition of the model's confidence.
    # =============================================================================
    joint_raw_contrib = {
        JOINT_NAMES[j]: float(joint_probs[j]) * unsafe_prob
        for j in range(NUM_JOINTS)
    }

    # Normalize contributions so that the sum equals the total unsafe
    # probability and express each contribution in percentage points (pp).
    total_raw = sum(joint_raw_contrib.values())

    joint_contrib_pp = {
        k: (v / total_raw) * unsafe_prob * 100
        for k, v in joint_raw_contrib.items()
    }

    # Sort joints from highest contribution to lowest contribution.
    sorted_contrib = sorted(
        joint_contrib_pp.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # =============================================================================
    # Rank Joints by Model-Predicted Risk
    # =============================================================================
    # Rank all joints according to the probabilities produced by the
    # joint_risk_head. The highest-ranked joint is considered the primary
    # contributor to the predicted risk.
    # =============================================================================
    ranked_joints = sorted(
        enumerate(JOINT_NAMES),
        key=lambda x: joint_probs[x[0]],
        reverse=True
    )

    # Store the complete explanation for every joint.
    explanation = []

    for jidx, joint_name in ranked_joints:

        # Human-readable joint name.
        display = JOINT_DISPLAY_NAMES.get(
            joint_name,
            joint_name.replace("_", " ").title()
        )

        # Probability assigned by the joint_risk_head.
        prob = float(joint_probs[jidx])

        # Integrated Gradients attribution for this joint.
        ig_score = ig_per_joint.get(joint_name, 0.0)

        # Actual measured value from the extracted biomechanical features.
        val = joint_values.get(joint_name)

        # Measurement unit for display.
        unit = JOINT_UNITS.get(joint_name, "")

        # Reference safe range for reporting purposes.
        safe_r = SAFE_RANGES_DISPLAY.get(joint_name)

        # Check whether the measured value falls outside the reference range.
        # This is displayed only for user interpretation and does NOT affect
        # the model's prediction.
        oor = (
            not (safe_r[0] <= val <= safe_r[1])
        ) if (safe_r and val is not None) else False

        # Contribution of this joint to the unsafe probability.
        contrib = joint_contrib_pp.get(joint_name, 0.0)

        # Save all explanation information for this joint.
        explanation.append({
            "body_part"        : display,
            "feature_key"      : joint_name,
            "model_prob"       : round(prob, 4),
            "ig_score"         : round(ig_score, 4),
            "unsafe_contrib_pp": round(contrib, 4),   # Percentage-point contribution
            "value"            : round(val, 2) if val is not None else None,
            "unit"             : unit,
            "safe_range"       : safe_r,
            "out_of_range"     : oor,
        })

    # The highest-ranked joint is identified as the primary cause of risk.
    # If no explanation exists, return "Unknown".
    primary_cause = explanation[0]["body_part"] if explanation else "Unknown"

    # ── Print report ───────────────────────────────────────────────────────────
    if verbose:
        print("=" * 72)
        print("YOGASAFE RISK EXPLANATION REPORT")
        print("=" * 72)
        print(f"  Video           : {os.path.basename(video_path)}")
        print(f"  Predicted Pose  : {predicted_pose}  ({p_prob[0][p_idx].item()*100:.1f}% confidence)")
        print(f"  Predicted Risk  : {predicted_risk.upper()}  ({r_prob[0][r_idx].item()*100:.1f}% confidence)")
        print()

        # ── Section 1: Joint risk ranking ─────────────────────────────────────
        print("  ─── JOINT RISK ASSESSMENT (ranked by model-predicted probability) ───")
        print("  Source: joint_risk_head — learned from labeled training data")
        print()

        for i, e in enumerate(explanation[:8], 1):
            val_str  = f"{e['value']}{e['unit']}" if e["value"] is not None else "N/A"
            oor_note = "  [outside reference range]" if e["out_of_range"] else ""
            ig_dir   = "↑ risky" if e["ig_score"] > 0 else "↓ safe"
            bar_len  = int(e["model_prob"] * 20)
            bar      = "█" * bar_len + "░" * (20 - bar_len)
            print(f"  #{i:<2} {e['body_part']:<30}")
            print(f"       Model prob        : {e['model_prob']:.4f}  [{bar}]")
            print(f"       IG support        : {e['ig_score']:+.4f}  {ig_dir}")
            print(f"       Unsafe contrib    : {e['unsafe_contrib_pp']:.4f}pp  (of {unsafe_prob*100:.2f}% total unsafe)")
            print(f"       Value             : {val_str}{oor_note}")
            if e["safe_range"]:
                lo, hi = e["safe_range"]; unit = e["unit"]
                print(f"       Ref range         : {lo}{unit} – {hi}{unit}  (annotation reference)")
            print()

        print(f"  PRIMARY CAUSE (model-identified) : {primary_cause}")
        print(f"  Model probability of risk         : {explanation[0]['model_prob']:.4f}")
        print()

        # ── Section 2: Unsafe signal decomposition ────────────────────────────
        print(f"  ─── WHERE DOES THE {unsafe_prob*100:.2f}% UNSAFE SIGNAL COME FROM? ───")
        print(f"  Even though predicted {predicted_risk.upper()} ({safe_prob*100:.2f}% confidence),")
        print(f"  the remaining {unsafe_prob*100:.2f}% unsafe probability is explained by:")
        print()

        top5_total = 0.0
        for joint_key, contrib_pp in sorted_contrib[:5]:
            display  = JOINT_DISPLAY_NAMES.get(joint_key, joint_key.replace("_", " ").title())
            share    = (contrib_pp / (unsafe_prob * 100)) * 100  # % share of the unsafe signal
            bar_len  = int(share / 5)                             # scale: 100% share = 20 blocks
            bar      = "█" * bar_len + "░" * (20 - bar_len)
            print(f"  {display:<30}  {contrib_pp:.4f}pp  [{bar}]  ({share:.1f}% of unsafe signal)")
            top5_total += contrib_pp

        print()
        print(f"  Top-5 joints account for : {top5_total:.4f}pp of {unsafe_prob*100:.2f}%")
        print(f"  Remaining {len(sorted_contrib)-5} joints account for : {(unsafe_prob*100 - top5_total):.4f}pp")
        print()
        print("  Interpretation:")
        top_joint_display = JOINT_DISPLAY_NAMES.get(sorted_contrib[0][0], sorted_contrib[0][0])
        print(f"    The model's residual doubt is driven primarily by {top_joint_display}.")
        print(f"    This joint has the highest learned risk probability AND contributes")
        print(f"    the most to the {unsafe_prob*100:.2f}% of the model's unsafe signal.")
        print()
        print("  Note: Rankings are determined entirely by the joint_risk_head output.")
        print("  Integrated Gradients scores provide a secondary interpretability check.")
        print("  Reference ranges shown for human context — not used during inference.")
        print("=" * 72)

    return {
        "Predicted Pose"    : predicted_pose,
        "Pose Confidence"   : f"{p_prob[0][p_idx].item()*100:.2f}%",
        "Predicted Risk"    : predicted_risk,
        "Risk Confidence"   : f"{r_prob[0][r_idx].item()*100:.2f}%",
        "Safe Probability"  : round(safe_prob * 100, 4),
        "Unsafe Probability": round(unsafe_prob * 100, 4),
        "Frames Processed"  : len(landmark_rows),
        "explanation"       : explanation,
        "primary_cause"     : primary_cause,
        "unsafe_decomposition": dict(sorted_contrib),
    }

print("explain_risk() defined — model-driven joint localization + unsafe decomposition ✓")
print("Usage: result = explain_risk('/content/crow.mp4')")


In [ ]:
## Step 17 – Batch Test with Risk Explanation + Timings

test_videos = [
    ("/content/boat1.mp4",           "boat unkn"),
    ("/content/chair1.mp4",          "Wchair unkw"),
    ("/content/crow1.mp4",           "crow unkw"),
    ("/content/downwarddog1.mp4",    "down dog unkw"),
    ("/content/halfmoon1.mp4",       "HALFMOn unkw"),
    ("/content/wheel1.mp4",          "wheel unkw"),
    ("/content/chairunsafe1.mp4",    "chair unkw"),
    ("/content/t(1)1.mp4",           "WAR1 unsafe/safe"),
    ("/content/w21.mp4",             "WAR2 safe"),
    ("/content/hm1.mp4",             "HALFMOON unsafe"),
    ("/content/IMG_43201.MOV",       "TRIANGLE unsafe"),
    ("/content/IMG_43211.MOV",       "WAR1 unsafe"),
    ("/content/IMG_43221.MOV",       "WAR1 safe"),
    ("/content/IMG_43231.MOV",       "WAR2 unsafe"),
    ("/content/IMG_4324.MOV",       "WAR2 safe"),
    ("/content/IMG_43251.MOV",       "TREE unsafe"),
    ("/content/IMG_4326.MOV",       "TREE safe"),
    ("/content/IMG_4328.mov",       "HALFMOON unsafe"),
    ("/content/IMG_43291.MOV",       "BOAT safe"),
]

all_timings = []

print("=" * 70)
print("BATCH INFERENCE WITH RISK EXPLANATION + TIMINGS")
print("=" * 70)

for video_path, expected in test_videos:
    if not os.path.exists(video_path):
        print(f"\n>>> SKIPPED (file not found): {video_path}")
        continue
    try:
        print(f"\n>>> Expected: {expected}")

        # ── PART 1: predict_video for timings ──────────────────────────
        pred = predict_video(video_path)

        if "error" in pred:
            print("  ERROR:", pred["error"])
            continue

        print(f"  Video            : {os.path.basename(video_path)}")
        print(f"  Predicted Pose   : {pred['Predicted Pose']}")
        print(f"  Pose Confidence  : {pred['Pose Confidence']}")
        print(f"  Predicted Risk   : {pred['Predicted Risk']}")
        print(f"  Risk Confidence  : {pred['Risk Confidence']}")
        print(f"  Frames Processed : {pred['Frames Processed']}")
        print(f"  Device           : {pred['Device']}")

        t = pred["timings"]
        print(f"\n  ⏱  Landmark Extraction   : {t['landmark_extraction_ms']:.1f} ms")
        print(f"  ⏱  Bio Feature Extraction: {t['bio_feature_extraction_ms']:.1f} ms")
        print(f"  ⏱  Preprocessing/Scaling : {t['preprocessing_ms']:.1f} ms")
        print(f"  ⏱  Tensor Transfer       : {t['tensor_transfer_ms']:.1f} ms")
        print(f"  ⏱  Model Inference (avg) : {t['model_inference_ms']:.3f} ms")
        print(f"  ⏱  Postprocessing        : {t['postprocessing_ms']:.3f} ms")
        print(f"  ⏱  TOTAL                 : {t['total_ms']:.1f} ms")

        if "gpu_memory_allocated_mb" in t:
            print(f"  🖥  GPU Memory Allocated  : {t['gpu_memory_allocated_mb']:.1f} MB")
            print(f"  🖥  GPU Memory Reserved   : {t['gpu_memory_reserved_mb']:.1f} MB")

        all_timings.append(t)

        # ── PART 2: explain_risk for body part report ───────────────────
        print()
        explain_risk(video_path, model=final_model, verbose=True)

    except Exception as e:
        import traceback
        print(f"  FAILED: {e}")
        traceback.print_exc()

# ── Timing summary ──────────────────────────────────────────────────────────
if all_timings:
    print("\n" + "=" * 70)
    print("HARDWARE INFERENCE SUMMARY (across all videos)")
    print("=" * 70)

    keys   = ["landmark_extraction_ms", "bio_feature_extraction_ms",
              "preprocessing_ms", "tensor_transfer_ms",
              "model_inference_ms", "postprocessing_ms", "total_ms"]
    labels = ["Landmark Extraction   ", "Bio Feature Extraction",
              "Preprocessing/Scaling ", "Tensor Transfer       ",
              "Model Inference (avg) ", "Postprocessing        ",
              "TOTAL                 "]

    for key, label in zip(keys, labels):
        vals = [t[key] for t in all_timings if key in t]
        if vals:
            print(f"  {label} | mean: {np.mean(vals):8.2f} ms "
                  f"| min: {np.min(vals):8.2f} ms "
                  f"| max: {np.max(vals):8.2f} ms")

print("\nDone.")

# 12. Deployment as an API Service

This section prepares the trained model for external usage through an API-based deployment approach.

FastAPI is used to create an inference endpoint that allows users or external applications to submit yoga videos for automated posture assessment.

Cloudflared is utilized to provide secure remote access to the API service during testing and demonstration.

In [ ]:
import os
# Install FastAPI and Cloudflare tunnel requirements
!pip install -q fastapi uvicorn python-multipart nest-asyncio

In [ ]:
# Install cloudflared
print("Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("Cloudflared installation complete.")

In [ ]:
# =============================================================================
# Import Required Libraries
# =============================================================================
# Import the libraries needed to build and expose the backend API,
# handle uploaded videos, start the web server, and create a public
# Cloudflare Tunnel for external access.

import nest_asyncio, uvicorn, tempfile, os, subprocess, threading, time, re
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

# Allows Uvicorn to run correctly inside environments like Google Colab or Jupyter.
nest_asyncio.apply()

# =============================================================================
# Create FastAPI Application
# =============================================================================
# Create a new FastAPI application.
# The variable name app2 is used to avoid conflicts with any previously
# created FastAPI application.
app2 = FastAPI()

# =============================================================================
# Enable Cross-Origin Resource Sharing (CORS)
# =============================================================================
# Allows requests from any frontend (HTML, React, etc.).
# This lets browsers communicate with the backend even when they are
# hosted on different domains or ports.
app2.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# =============================================================================
# Prediction Endpoint
# =============================================================================
# Receives an uploaded yoga video and returns:
#   • Pose prediction
#   • Risk prediction
#   • Joint-level explanation
#   • Performance timings
@app2.post("/predict")
async def predict(file: UploadFile = File(...)):

    # Preserve the uploaded file extension (.mp4, .mov, etc.).
    suffix = os.path.splitext(file.filename)[-1] or ".mp4"

    # Save the uploaded file temporarily on disk.
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    try:

        # ==============================================================
        # Step 1: Run the prediction pipeline
        # ==============================================================
        # Predict:
        #   • Yoga pose
        #   • Safe/unsafe classification
        #   • Processing timings
        pred = predict_video(tmp_path)

        # Return immediately if prediction failed.
        if "error" in pred:
            return pred

        # ==============================================================
        # Step 2: Generate explainability results
        # ==============================================================
        # Produce:
        #   • Joint risk probabilities
        #   • Safe/unsafe probabilities
        #   • Primary risky joint
        #   • Unsafe signal decomposition
        explain = explain_risk(tmp_path, model=final_model, verbose=False)

        # If explanation fails, still return prediction results.
        if "error" in explain:
            return {**pred, "explain_error": explain["error"]}

        # ==============================================================
        # Combine Prediction + Explainability Results
        # ==============================================================
        # Merge outputs from predict_video() and explain_risk()
        # into a single JSON response for the frontend.
        return {
            "Predicted Pose"      : pred.get("Predicted Pose"),
            "Pose Confidence"     : pred.get("Pose Confidence"),
            "Predicted Risk"      : pred.get("Predicted Risk"),
            "Risk Confidence"     : pred.get("Risk Confidence"),
            "Frames Processed"    : pred.get("Frames Processed"),
            "Device"              : pred.get("Device"),
            "timings"             : pred.get("timings"),

            # Explanation fields used by the HTML report/dashboard.
            "Safe Probability"    : explain.get("Safe Probability"),
            "Unsafe Probability"  : explain.get("Unsafe Probability"),
            "explanation"         : explain.get("explanation"),
            "primary_cause"       : explain.get("primary_cause"),
            "unsafe_decomposition": explain.get("unsafe_decomposition"),
        }

    finally:
        # Always remove the temporary uploaded video after processing
        # to prevent unnecessary disk usage.
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

# =============================================================================
# Health Check Endpoint
# =============================================================================
# Simple endpoint used to verify that the backend server is running.
@app2.get("/health")
def health():
    return {"status": "ok"}

# =============================================================================
# Start FastAPI Server
# =============================================================================
# Runs the FastAPI application on port 8001.
# Port 8001 is used to avoid conflicts with other servers that may
# already be running on port 8000.
def start_uvicorn():
    uvicorn.run(app2, host="0.0.0.0", port=8001)

# Start the server in a background thread so the notebook can continue executing.
threading.Thread(target=start_uvicorn, daemon=True).start()

# Wait briefly to allow the server to start.
time.sleep(1)

# =============================================================================
# Start Cloudflare Tunnel
# =============================================================================
# Launch Cloudflare Tunnel to expose the local backend server
# to the public internet through a temporary HTTPS URL.
print("Starting Cloudflare Tunnel on port 8001...")

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Continuously monitor Cloudflare's output until the public URL is generated.
for line in proc.stdout:

    decoded_line = line.decode()

    # Search for the generated TryCloudflare URL.
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", decoded_line)

    if match:
        url = match.group(0)

        print("\n" + "=" * 60)
        print("YOGASAFE BACKEND IS LIVE (port 8001)!")
        print(f"API URL: {url}")
        print("Paste this URL into your HTML site.")
        print("=" * 60)

        # Stop reading once the public URL has been found.
        break

## Instructions for Running the Notebook

If you are running this notebook for the first time, or if your Colab runtime has just started/restarted, please run all cells sequentially from the beginning.

### If you have restarted the Colab runtime and wish to continue from a saved state:

1.  **Run the initial setup cells:**
    *   Cell #1 (Install protobuf, MediaPipe, and download model) - _Requires runtime restart after running_.
    *   Cell #2 (Mount Google Drive, define paths).
    *   Cell #3 (Import libraries, print versions).
    *   Cell #4 (Define device and hyperparameters).
    *   Cell #oos7QJ0FJpZD (MultiTaskYogaTransformer model definition).

2.  **Then, run Cell #BknVbDjoe1Rn:** This cell is titled 'RESTORE FROM GOOGLE DRIVE' and will load all necessary artifacts (scalers, label encoders, and the trained model) from your Google Drive, allowing you to skip the data loading, feature extraction, and training steps.

3.  After restoring, you can proceed directly to cells like 'Step 14 – predict_video' and 'Step 15 – explain_risk()' for inference, or 'Step 17 – Batch Test with Risk Explanation + Timings' to test the model.

In [ ]:
import os
import shutil
from google.colab import drive

# Ensure Google Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the target directory in Google Drive
# This will be 'YogaModel_SavedArtifacts' in your 'My Drive' folder
drive_save_dir = os.path.join('/content/drive/MyDrive', os.path.basename(OUTPUT_ROOT))
os.makedirs(drive_save_dir, exist_ok=True)

print(f"Saving all contents from '{OUTPUT_ROOT}' to '{drive_save_dir}'")

# Copy all files and subdirectories from OUTPUT_ROOT to Google Drive
for item in os.listdir(OUTPUT_ROOT):
    s = os.path.join(OUTPUT_ROOT, item)
    d = os.path.join(drive_save_dir, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
        print(f"  Copied directory: {item}")
    else:
        shutil.copy2(s, d)
        print(f"  Copied file: {item}")

print("All generated artifacts have been saved to Google Drive.")
#11

## Optional – Resume From a Previous Session

Use this section instead of Steps 1–13, not in addition.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔄  RESTORE FROM GOOGLE DRIVE  (run this after a session reset)
# Run AFTER: cell #3 (imports), cell #4 (hyperparams/device), and cell #oos7QJ0FJpZD (MultiTaskYogaTransformer model definition).
# Skip ALL training / data-loading cells — go straight to FastAPI
# ══════════════════════════════════════════════════════════════

import os, shutil, joblib, torch
import numpy as np
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# ── 1. Mount Drive ────────────────────────────────────────────
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

# ── 2. Restore artifact folder from Drive → /content/ ────────
DRIVE_ARTIFACTS = "/content/drive/MyDrive/YogaModel_SavedArtifacts"
OUTPUT_ROOT     = "/content/YogaModel_SavedArtifacts"

if not os.path.exists(OUTPUT_ROOT):
    print(f"Copying artifacts from Drive …")
    shutil.copytree(DRIVE_ARTIFACTS, OUTPUT_ROOT)
    print("Done.")
else:
    print("Artifacts already present locally, skipping copy.")

MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")

# ── 3. Load scalers + feature_cols ───────────────────────────
feature_scaler  = joblib.load(os.path.join(MODEL_DIR, "bio_feature_scaler.pkl"))
landmark_scaler = joblib.load(os.path.join(MODEL_DIR, "landmark_scaler.pkl"))
biomech_scaler  = joblib.load(os.path.join(MODEL_DIR, "biomech_scaler.pkl"))
feature_cols    = joblib.load(os.path.join(MODEL_DIR, "bio_feature_cols.pkl"))
print(f"Scalers loaded  |  feature_cols: {len(feature_cols)}")

# ── 4. Load model checkpoint ──────────────────────────────────
MODEL_PATH = os.path.join(MODEL_DIR, "multitask_yoga_transformer.pth")
ckpt = torch.load(MODEL_PATH, map_location=device)

# ── 5. Rebuild label encoders from checkpoint metadata ────────
pose_encoder = LabelEncoder()
risk_encoder = LabelEncoder()
pose_encoder.classes_ = np.array(ckpt["pose_classes"])
risk_encoder.classes_ = np.array(ckpt["risk_classes"])
JOINT_NAMES = ckpt["joint_names"]
NUM_JOINTS  = len(JOINT_NAMES)
print(f"Pose classes : {list(pose_encoder.classes_)}")
print(f"Risk classes : {list(risk_encoder.classes_)}")
print(f"Joint names  : {JOINT_NAMES}")

# ── 6. Rebuild + load the model ───────────────────────────────
final_model = MultiTaskYogaTransformer(
    landmark_dim = ckpt["landmark_dim"],
    biomech_dim  = ckpt["biomech_dim"],
    agg_dim      = ckpt["agg_dim"],
    num_poses    = ckpt["num_poses"],
    num_risks    = ckpt["num_risks"],
    num_joints   = ckpt["num_joints"],
    d_model      = 256,
    nhead        = 8,
    num_layers   = 4,
    dropout      = 0.30,
).to(device)

final_model.load_state_dict(ckpt["model_state_dict"])
final_model.eval()
print("✅  final_model loaded and ready for inference.")

## Deployment – Serve Model as API

In [ ]:
import os
# Install FastAPI and Cloudflare tunnel requirements
!pip install -q fastapi uvicorn python-multipart nest-asyncio

In [ ]:
# Install cloudflared
print("Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("Cloudflared installation complete.")

In [ ]:


import nest_asyncio, uvicorn, tempfile, os, subprocess, threading, time, re
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()

app2 = FastAPI()   # new variable name avoids conflict with old `app`
app2.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app2.post("/predict")
async def predict(file: UploadFile = File(...)):
    suffix = os.path.splitext(file.filename)[-1] or ".mp4"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name
    try:
        # Step 1: predict_video → pose, risk, timings
        pred = predict_video(tmp_path)
        if "error" in pred:
            return pred

        # Step 2: explain_risk → joint probs, safe/unsafe split, decomp
        explain = explain_risk(tmp_path, model=final_model, verbose=False)
        if "error" in explain:
            return {**pred, "explain_error": explain["error"]}

        # Merge into one response
        return {
            "Predicted Pose"      : pred.get("Predicted Pose"),
            "Pose Confidence"     : pred.get("Pose Confidence"),
            "Predicted Risk"      : pred.get("Predicted Risk"),
            "Risk Confidence"     : pred.get("Risk Confidence"),
            "Frames Processed"    : pred.get("Frames Processed"),
            "Device"              : pred.get("Device"),
            "timings"             : pred.get("timings"),
            # explain_risk fields — feed the HTML report
            "Safe Probability"    : explain.get("Safe Probability"),
            "Unsafe Probability"  : explain.get("Unsafe Probability"),
            "explanation"         : explain.get("explanation"),
            "primary_cause"       : explain.get("primary_cause"),
            "unsafe_decomposition": explain.get("unsafe_decomposition"),
        }
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

@app2.get("/health")
def health():
    return {"status": "ok"}

# Run on port 8001 — avoids conflict with old server on 8000
def start_uvicorn():
    uvicorn.run(app2, host="0.0.0.0", port=8001)

threading.Thread(target=start_uvicorn, daemon=True).start()
time.sleep(1)

print("Starting Cloudflare Tunnel on port 8001...")
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

for line in proc.stdout:
    decoded_line = line.decode()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", decoded_line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 60)
        print("YOGASAFE BACKEND IS LIVE (port 8001)!")
        print(f"API URL: {url}")
        print("Paste this URL into your HTML site.")
        print("=" * 60)
        break

# Final Implementation Summary

This notebook demonstrates the complete development lifecycle of the Yoga Posture Assessment System, from raw video processing to AI-powered posture evaluation.

The implemented pipeline integrates computer vision-based pose estimation, biomechanical analysis, Transformer-based deep learning, explainable assessment, and API deployment.

The results obtained from this implementation serve as the experimental foundation for evaluating the proposed thesis system.